# 02 · Build site

Turns `data/site_data.pkl` from `01_build_data` into the site's pages, written to the `site` folder next to `League`. Each page holds both layouts: the desktop one and the phone one, and the browser shows whichever fits the screen.

Pages written: the league home (`index.html`), the start page (`start.html`), the weekly recap, one page per matchup, standings, one page per team, Draft & Moves, and History. The menu links to the Almost Points page that FantasyLuck builds in `site/almost_points`.

In [ ]:
from pathlib import Path
import math, re, statistics
import pandas as pd

DATA_FILE = Path("data/site_data.pkl")
SITE = Path("../site")
SITE.mkdir(exist_ok=True)

D = pd.read_pickle(DATA_FILE)
globals().update(D["league"])
BENCHW = BENCH                     # the Teams page calls the bench numbers BENCHW
print(LEAGUE_NAME, "· Week", WEEK, "·", len(D["teams"]), "teams ·", len(D["matchups"]), "matchups")

## Shared look

Colors, fonts, icons, the wave dividers, and the start page. Everything else builds on these.

In [ ]:
F="'Alegreya Sans SC', 'Trebuchet MS', sans-serif"

INK="#1B2533"; INK2="#5B6266"; INK3="#868D91"; LINE="#DADDDF"; LINE2="#EEF0F2"; NAVY="#13294B"; WOLF="#A5ACAF"; WOLF_L="#E6E8E9"

GREEN="#1F8A4C"; RED="#C9302C"

card=lambda inner,extra="": f'<section style="background: #FFFFFF; border: 1px solid {LINE}; border-radius: 8px; padding: 18px 20px{extra}">{inner}</section>'

h2=lambda t,right="": f'<div style="display: flex; justify-content: space-between; align-items: baseline; gap: 12px; margin-bottom: 12px"><h2 style="margin: 0; font-size: 18px; font-weight: 800">{t}</h2>{right}</div>'

INSIDE=[("Recap","Every matchup, the week&#39;s top and bottom scorers, closest game, biggest blowout, and the weekly awards."),
        ("Standings","Your record next to the all-play record and record vs. the median, plus your record with anyone else&#39;s schedule."),
        ("Teams","One page per manager: weekly scores, rank each week, where the points came from, and points left on the bench."),
        ("Almost Points","Points lost to flags, missed throws, drops, replay, and goal-line stops, for every player and every team."),
        ("Draft &amp; Moves","What every draft pick, trade, and waiver pickup has scored since."),
        ("History","All-time head-to-head records and the league record book.")]

# ---------------- Home, no league (desktop) ----------------
ICONS={
 "Recap":'<path d="M4 5h16v14H4z M8 9h8 M8 13h8 M8 17h5"/>',
 "Standings":'<path d="M5 20V10 M12 20V4 M19 20v-7"/>',
 "Teams":'<circle cx="9" cy="8" r="3.2"/><circle cx="17" cy="9" r="2.6"/><path d="M3 20c0-3.5 2.7-6 6-6s6 2.5 6 6 M15 14.5c3 0 6 2 6 5.5"/>',
 "Almost Points":'<path d="M3 17l5-5 4 4 8-9"/><path d="M15 7h5v5"/>',
 "Draft &amp; Moves":'<path d="M4 8h13l-3-3 M20 16H7l3 3"/>',
 "History":'<circle cx="12" cy="12" r="8"/><path d="M12 7v5l3 2"/>',
}

def icon(t,c=NAVY,s=26):
    return f'<svg width="{s}" height="{s}" viewBox="0 0 24 24" fill="none" stroke="{c}" stroke-width="1.9" stroke-linecap="round" stroke-linejoin="round" aria-hidden="true">{ICONS[t]}</svg>'

def waves(h=120, top="#13294B", layers=("#A5ACAF", "#E6E8E9", "#FFFFFF")):
    # three layered swells: wolf grey, light grey, then the color of the section below
    a, b, c = layers
    return (f'<svg viewBox="0 0 1440 {h}" preserveAspectRatio="none" aria-hidden="true" style="display: block; width: 100%; height: {h}px; background: {top}">'
            f'<path d="M0 {h*0.45:.0f} C 180 {h*0.05:.0f}, 360 {h*0.05:.0f}, 540 {h*0.4:.0f} S 900 {h*0.8:.0f}, 1080 {h*0.4:.0f} S 1340 {h*0.05:.0f}, 1440 {h*0.3:.0f} V {h} H 0 Z" fill="{a}" opacity="0.55"/>'
            f'<path d="M0 {h*0.65:.0f} C 240 {h*0.3:.0f}, 420 {h*0.35:.0f}, 620 {h*0.62:.0f} S 980 {h*0.9:.0f}, 1180 {h*0.6:.0f} S 1380 {h*0.38:.0f}, 1440 {h*0.5:.0f} V {h} H 0 Z" fill="{b}"/>'
            f'<path d="M0 {h*0.82:.0f} C 200 {h*0.62:.0f}, 470 {h*0.66:.0f}, 720 {h*0.82:.0f} S 1180 {h*0.98:.0f}, 1440 {h*0.74:.0f} V {h} H 0 Z" fill="{c}"/></svg>')

def tick(a,who,val):
    return (f'<span style="display: inline-flex; gap: 8px; align-items: baseline; white-space: nowrap">'
            f'<span style="font-size: 12px; font-weight: 800; letter-spacing: 1px; color: {NAVY}; background: {WOLF}; padding: 2px 7px; border-radius: 3px">{a}</span>'
            f'<b style="color: #FFFFFF; font-size: 16px">{who}</b><span style="color: #D5D9DB; font-size: 15px">{val}</span>'
            f'<span aria-hidden="true" style="color: #3C5476; padding: 0 14px">◆</span></span>')

ticker="".join(tick(*t) for t in TICK)

def row(a,b,col=INK):
    return f'<div style="display: flex; justify-content: space-between; gap: 10px"><span style="color: {INK2}">{a}</span><b style="color: {col}">{b}</b></div>'

report=(f'<div style="transform: rotate(-2.5deg); background: #FFFFFF; color: {INK}; border-radius: 12px; box-shadow: 0 18px 40px rgba(5,15,35,0.45); padding: 20px 22px; display: flex; flex-direction: column; gap: 12px; width: 100%; max-width: 400px; box-sizing: border-box">'
        f'<div style="display: flex; justify-content: space-between; align-items: baseline; border-bottom: 2px solid {NAVY}; padding-bottom: 8px"><b style="font-size: 20px; color: {NAVY}">The Week {WEEK} Report</b><span style="font-size: 13px; color: {INK3}">{LEAGUE_NAME}</span></div>'
        + row("Top score",f"[Team {HI10}] {SC10D[HI10]:.1f}",GREEN) + row("Closest game",f"{CLOSE10[1]:.1f} to {CLOSE10[3]:.1f}") + row("Unluckiest team",f"[Team {unl}] {tl[unl]:.1f} lost",RED) + row("Most points in a loss",f"[Team {ML10[2]}] {ML10[3]:.1f}")
        + f'<div style="display: flex; gap: 3px; height: 10px; border-radius: 5px; overflow: hidden; margin-top: 2px"><span style="flex: 151.8 1 0; background: {GREEN}"></span><span style="flex: 40 1 0; background: #9F1B17"></span><span style="flex: 22 1 0; background: #E5484D"></span><span style="flex: 12 1 0; background: #F2837F"></span></div></div>')

TAG={"Recap":"The week, settled","Standings":"Your record, and the one you earned","Teams":"Every manager, up close",
     "Almost Points":"The points that almost counted","Draft &amp; Moves":"Every pick, graded in hindsight","History":"Grudges, on the record"}

def bigcard(i,t,d):
    note = "Open now, no league needed →" if t=="Almost Points" else "Needs your league"
    return (f'<a href="{"almost_points/index.html" if t=="Almost Points" else "#top"}" style="display: flex; gap: 18px; align-items: flex-start; padding: 22px; background: #FFFFFF; border: 1px solid {LINE}; border-radius: 10px; text-decoration: none; color: {INK}">'
            f'<span style="flex: none; display: inline-flex; width: 68px; height: 68px; align-items: center; justify-content: center; border-radius: 50%; background: {NAVY}">{icon(t, "#FFFFFF", 34)}</span>'
            f'<span style="display: flex; flex-direction: column; gap: 4px"><span style="font-size: 22px; font-weight: 800; color: {NAVY}; line-height: 1.05">{t}</span>'
            f'<span style="font-size: 16px; font-weight: 700; color: {INK2}">{TAG[t]}</span>'
            f'<span style="font-size: 15px; color: {INK2}; line-height: 1.4; margin-top: 4px">{d}</span>'
            f'<span style="font-size: 13px; font-weight: 700; color: {GREEN if t=="Almost Points" else INK3}; margin-top: 6px">{note}</span></span></a>')

cards="".join(bigcard(i,t,d) for i,(t,d) in enumerate(INSIDE,1))

def step(n,t,d=None):
    return f'<div style="display: flex; gap: 14px; align-items: center; background: #FFFFFF; border-radius: 10px; padding: 16px 18px"><span style="flex: none; display: inline-flex; width: 48px; height: 48px; align-items: center; justify-content: center; border-radius: 50%; background: {NAVY}; color: #FFFFFF; font-size: 24px; font-weight: 800">{n}</span><b style="font-size: 20px; color: {NAVY}">{t}</b></div>'

def main_html():
    return f'''<div style="min-height: 100%; background: #FFFFFF; color: {INK}; font-family: {F}; font-size: 15px">
{navbar(locked=True)}
<section style="position: relative; overflow: hidden; background: {NAVY}; color: #FFFFFF">
<span aria-hidden="true" style="position: absolute; right: -30px; top: -70px; font-size: 380px; font-weight: 800; line-height: 1; color: rgba(165,172,175,0.08); letter-spacing: -6px">WFR</span>
<div style="position: relative; max-width: 1240px; margin: 0 auto; padding: 60px 24px 30px; display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 48px; align-items: center">
<div style="display: flex; flex-direction: column; gap: 18px">
<img src="assets/wfr-lockup.png" alt="Wave Fantasy Report" style="height: 110px; width: auto; align-self: flex-start; margin-bottom: 8px">
<h1 style="margin: 0; font-size: 64px; line-height: 0.98; font-weight: 800">Every point, every pick, every bad beat.</h1>
<p style="margin: 0; font-size: 19px; color: #D5D9DB; line-height: 1.45; max-width: 540px">Your league&#39;s season on the record: the matchups, the standings behind the standings, the points that slipped away, and how every draft pick and pickup turned out. Nothing to download and nothing to sign into.</p>
<form style="display: flex; flex-direction: column; gap: 10px; margin-top: 6px; max-width: 540px">
<div role="tablist" aria-label="Platform" style="display: inline-flex; align-self: flex-start; gap: 2px; padding: 3px; border-radius: 9px; background: rgba(255,255,255,0.08); border: 1px solid #3C5476"><button style="font: inherit; font-size: 15px; font-weight: 800; min-height: 38px; padding: 0 18px; border: 0; border-radius: 7px; background: {WOLF}; color: {NAVY}">Sleeper</button><button style="font: inherit; font-size: 15px; font-weight: 700; min-height: 38px; padding: 0 18px; border: 0; border-radius: 7px; background: transparent; color: #FFFFFF">ESPN</button></div>
<div style="display: flex; gap: 10px">
<input id="leagueId" inputmode="numeric" placeholder="[YOUR LEAGUE ID]" style="flex: 1; min-width: 0; font: inherit; font-size: 17px; padding: 13px 14px; border: 0; border-radius: 8px; background: #FFFFFF; color: {INK}">
<a href="index.html" style="display: inline-flex; align-items: center; padding: 0 22px; min-height: 50px; border-radius: 8px; background: {WOLF}; color: {NAVY}; text-decoration: none; font-weight: 800; font-size: 17px">Open my league →</a>
</div>
<p style="margin: 0; font-size: 14px; color: #BFC5C8">In the Sleeper app: your league, then Settings, then General.</p>
<div style="display: flex; gap: 10px; align-items: flex-start; padding: 12px 14px; border-radius: 8px; background: rgba(255,255,255,0.06); border: 1px solid #3C5476; font-size: 14px; color: #D5D9DB; line-height: 1.45"><span style="flex: none; font-size: 11px; font-weight: 800; letter-spacing: 1px; padding: 2px 7px; border-radius: 4px; background: {WOLF}; color: {NAVY}; margin-top: 1px">ESPN</span><span>Use the number after leagueId= in your league&#39;s web address. The league has to be viewable to the public for WFR to sync: your commissioner can turn that on in League Settings, under Basic Settings. Private ESPN leagues can&#39;t sync.</span></div>
</form>
</div>
<div style="display: flex; justify-content: center">{report}</div>
</div>
{waves(110, NAVY)}
</section>
<main style="max-width: 1240px; margin: 0 auto; padding: 10px 24px 48px; display: flex; flex-direction: column; gap: 40px">
<section style="display: flex; flex-direction: column; gap: 18px">
<div style="display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 16px">{cards}</div>
</section>
</main>
<section style="background: {WOLF_L}">
<div style="max-width: 1240px; margin: 0 auto; padding: 40px 24px; display: flex; flex-direction: column; gap: 22px">
<h2 style="margin: 0; font-size: 30px; font-weight: 800; color: {NAVY}">How it works</h2>
<div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 16px">
{step(1,"Enter your league ID")}
{step(2,"Send the link")}
{step(3,"Check back")}
</div></div></section>
<footer style="background: {NAVY}; color: #FFFFFF">
{waves(110, WOLF_L, ("#A5ACAF", "#3C5476", NAVY))}
<div style="max-width: 1240px; margin: 0 auto; padding: 10px 24px 40px; display: flex; justify-content: space-between; align-items: flex-end; gap: 24px; flex-wrap: wrap">
<div style="display: flex; flex-direction: column; gap: 6px"><img src="assets/wfr-lockup.png" alt="Wave Fantasy Report" style="height: 96px; width: auto; display: block"></div>
<div style="display: flex; flex-direction: column; gap: 6px; align-items: flex-end; text-align: right"><span style="font-size: 18px; font-weight: 700">New numbers every Wednesday.</span></div>
</div></footer>
</div>'''

# ---------------- phone, no league ----------------
def mcard(t,d):
    return (f'<a href="#" style="display: flex; gap: 14px; align-items: flex-start; padding: 16px 14px; border-bottom: 1px solid {LINE2}; text-decoration: none; color: {INK}">'
            f'<span style="flex: none; display: inline-flex; width: 52px; height: 52px; align-items: center; justify-content: center; border-radius: 50%; background: {NAVY}">{icon(t, "#FFFFFF", 26)}</span>'
            f'<span style="display: flex; flex-direction: column; gap: 2px"><span style="font-size: 19px; font-weight: 800; color: {NAVY}">{t}</span>'
            f'<span style="font-size: 15px; font-weight: 700; color: {INK2}">{TAG[t]}</span><span style="font-size: 14px; color: {INK2}; line-height: 1.4; margin-top: 2px">{d}</span></span></a>')

mcards="".join(mcard(t,d) for t,d in INSIDE)

def pm_html():
    return f'''<div style="width: 390px; min-height: 2700px; background: #FFFFFF; color: {INK}; font-family: {F}; font-size: 15px">
{navbar(locked=True, mobile=True)}
<section style="position: relative; overflow: hidden; background: {NAVY}; color: #FFFFFF">
<span aria-hidden="true" style="position: absolute; right: -20px; top: -24px; font-size: 170px; font-weight: 800; line-height: 1; color: rgba(165,172,175,0.09)">WFR</span>
<div style="position: relative; padding: 30px 16px 18px; display: flex; flex-direction: column; gap: 14px">
<img src="assets/wfr-lockup.png" alt="Wave Fantasy Report" style="height: 72px; width: auto; align-self: flex-start">
<h1 style="margin: 0; font-size: 42px; line-height: 0.98; font-weight: 800">Every point, every pick, every bad beat.</h1>
<p style="margin: 0; font-size: 16px; color: #D5D9DB; line-height: 1.45">Your league&#39;s season on the record: the matchups, the standings behind the standings, the points that slipped away, and how every pick turned out.</p>
<div role="tablist" aria-label="Platform" style="display: inline-flex; align-self: flex-start; gap: 2px; padding: 3px; border-radius: 9px; background: rgba(255,255,255,0.08); border: 1px solid #3C5476"><button style="font: inherit; font-size: 15px; font-weight: 800; min-height: 38px; padding: 0 18px; border: 0; border-radius: 7px; background: {WOLF}; color: {NAVY}">Sleeper</button><button style="font: inherit; font-size: 15px; font-weight: 700; min-height: 38px; padding: 0 18px; border: 0; border-radius: 7px; background: transparent; color: #FFFFFF">ESPN</button></div>
<input id="leagueIdM" inputmode="numeric" placeholder="[YOUR LEAGUE ID]" style="font: inherit; font-size: 17px; padding: 13px 14px; border: 0; border-radius: 8px; background: #FFFFFF; color: {INK}">
<a href="index.html" style="display: flex; justify-content: center; align-items: center; min-height: 50px; border-radius: 8px; background: {WOLF}; color: {NAVY}; text-decoration: none; font-weight: 800; font-size: 17px">Open my league →</a>
<p style="margin: 0; font-size: 14px; color: #BFC5C8">In the Sleeper app: your league, then Settings, then General.</p>
<div style="display: flex; gap: 10px; align-items: flex-start; padding: 12px 14px; border-radius: 8px; background: rgba(255,255,255,0.06); border: 1px solid #3C5476; font-size: 14px; color: #D5D9DB; line-height: 1.45"><span style="flex: none; font-size: 11px; font-weight: 800; letter-spacing: 1px; padding: 2px 7px; border-radius: 4px; background: {WOLF}; color: {NAVY}; margin-top: 1px">ESPN</span><span>Use the number after leagueId= in your league&#39;s web address. The league has to be viewable to the public for WFR to sync: your commissioner can turn that on in League Settings, under Basic Settings. Private ESPN leagues can&#39;t sync.</span></div>
</div>
<div style="position: relative; padding: 10px 22px 28px; display: flex; justify-content: center">{report}</div>
{waves(70, NAVY)}
</section>
<main style="padding: 8px 16px 40px; display: flex; flex-direction: column; gap: 22px">
<section><div style="border: 1px solid {LINE}; border-radius: 10px">{mcards}</div></section>
<a href="almost_points/index.html" style="font-weight: 700">Open Almost Points, no league needed →</a>
</main>
<section style="background: {WOLF_L}; padding: 28px 16px; display: flex; flex-direction: column; gap: 12px">
<h2 style="margin: 0; font-size: 24px; font-weight: 800; color: {NAVY}">How it works</h2>
{step(1,"Enter your league ID")}
{step(2,"Send the link")}
{step(3,"Check back")}
</section>
<footer style="background: {NAVY}; color: #FFFFFF">{waves(70, WOLF_L, ("#A5ACAF", "#3C5476", NAVY))}
<div style="padding: 6px 16px 30px; display: flex; flex-direction: column; gap: 6px"><img src="assets/wfr-lockup.png" alt="Wave Fantasy Report" style="height: 78px; width: auto; display: block">
<span style="font-size: 16px; font-weight: 700; margin-top: 8px">New numbers every Wednesday.</span></div></footer>
</div>'''

## Page shell, menu, and links

`page` wraps a desktop layout and a phone layout into one file and switches between them at 760 pixels wide. The menu button opens the navigation drawer, which links every page. `write_page` swaps every `[Team X]` placeholder for the team's name before saving.

File names: `team-<id>.html` for each team, `recap-week-<n>.html` for each earlier week this season (the latest is `recap.html`), `recap-<season>-week-<n>.html` for past seasons, and `matchup-<season>-week-<n>-<id>-<id>.html` for each game, using ESPN's team IDs so they never change when a team renames itself.

In [ ]:
ordn = lambda n: f"{n}{'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')}"
team_file = lambda t: f"team-{AB2ID[t]}.html"
# Every week of every season gets its own recap and matchup pages; this week's recap is recap.html
def recap_file(y, w):
    if y == SEASON:
        return "recap.html" if w == WEEK else f"recap-week-{w}.html"
    return f"recap-{y}-week-{w}.html"
matchup_file = lambda a, b, w, y=SEASON: f"matchup-{y}-week-{w}-{AB2ID[a]}-{AB2ID[b]}.html"
# The season picker opens each season's last week; the week picker lists the weeks of the season on screen
season_options = lambda: "".join(f'<option value="{recap_file(y, max(w for s, w in RECAPS if s == y))}"{" selected" if y == RS else ""}>{y}</option>'
                                  for y in sorted({s for s, _ in RECAPS}, reverse=True))
week_options = lambda: "".join(f'<option value="{recap_file(RS, w)}"{" selected" if w == RW else ""}>Week {w}</option>'
                               for w in sorted((w for s, w in RECAPS if s == RS), reverse=True))
RS, RW, HAS_LUCK = SEASON, WEEK, True

NAV = [("Recap", "recap.html"), ("Standings", "standings.html"), ("Teams", "teams.html"),
       ("Almost Points", "almost_points/index.html"), ("Draft &amp; Moves", "draft.html"), ("History", "history.html")]

def navbar(active=None, locked=False, mobile=False, league=False):
    # No top bar, just a menu button that opens the navigation drawer
    btn = (f'<button type="button" onclick="openMenu()" aria-label="Open menu" style="display: inline-flex; width: 46px; height: 46px; align-items: center; justify-content: center; border-radius: 50%; background: rgba(255,255,255,0.08); border: 1px solid #3C5476; cursor: pointer; padding: 0">'
           f'<svg width="22" height="22" viewBox="0 0 22 22" aria-hidden="true"><path d="M3 6h16M3 11h16M3 16h16" stroke="#FFFFFF" stroke-width="2" fill="none" stroke-linecap="round"/></svg></button>')
    if mobile:
        return f'<div style="background: {NAVY}; padding: 12px 16px 0">{btn}</div>'
    return f'<div style="background: {NAVY}"><div style="max-width: 1240px; margin: 0 auto; padding: 16px 24px 0">{btn}</div></div>'

def drawer(active):
    items = "".join(f'''<a href="{h}" style="display: flex; align-items: center; gap: 14px; padding: 14px 18px; border-radius: 10px; text-decoration: none; background: {'rgba(255,255,255,0.10)' if t == active else 'transparent'}; color: {'#FFFFFF' if t == active else '#D5D9DB'}">
{icon(t, "#FFFFFF" if t == active else WOLF, 24)}<span style="font-size: 20px; font-weight: {800 if t == active else 600}">{t}</span>{'<span style="margin-left: auto; width: 4px; height: 22px; border-radius: 2px; background: ' + WOLF + '"></span>' if t == active else ''}</a>''' for t, h in NAV)
    return f'''<div id="menu" class="drawer" onclick="if (event.target === this) closeMenu()">
<nav aria-label="Site" style="position: absolute; left: 0; top: 0; bottom: 0; width: min(320px, 86vw); background: {NAVY}; box-shadow: 8px 0 30px rgba(5,15,35,0.35); display: flex; flex-direction: column; padding: 18px 14px 24px; box-sizing: border-box; overflow-y: auto; font-family: {F}">
<img src="assets/wfr-lockup.png" alt="Wave Fantasy Report" style="height: 52px; width: auto; align-self: flex-start; margin: 0 4px 16px">
<div style="display: flex; justify-content: space-between; align-items: center; padding: 0 4px 14px">
<a href="index.html" style="color: #FFFFFF; text-decoration: none; display: flex; flex-direction: column"><span style="font-size: 13px; color: {WOLF}">League</span><span style="font-size: 20px; font-weight: 800">{LEAGUE_NAME}</span></a>
<button type="button" onclick="closeMenu()" aria-label="Close menu" style="display: inline-flex; width: 44px; height: 44px; align-items: center; justify-content: center; border-radius: 50%; border: 1px solid #3C5476; background: transparent; cursor: pointer"><svg width="18" height="18" viewBox="0 0 18 18" aria-hidden="true"><path d="M4 4l10 10M14 4L4 14" stroke="#FFFFFF" stroke-width="2" stroke-linecap="round"/></svg></button></div>
<a href="index.html" style="display: flex; align-items: center; gap: 14px; padding: 14px 18px; color: #D5D9DB; text-decoration: none; font-size: 18px; font-weight: 600; border-bottom: 1px solid #23395C; margin-bottom: 8px">Home</a>
<div style="display: flex; flex-direction: column; gap: 4px">{items}</div>
<div style="margin-top: auto; padding: 18px 4px 0; display: flex; flex-direction: column; gap: 8px">
<button type="button" style="font: inherit; min-height: 44px; border-radius: 8px; border: 0; background: {WOLF}; color: {NAVY}; font-weight: 800; cursor: pointer">Copy league link</button>
<a href="start.html" style="display: flex; justify-content: center; align-items: center; min-height: 44px; border-radius: 8px; border: 1px solid #3C5476; color: #FFFFFF; text-decoration: none">Change league</a></div></nav></div>'''

SCRIPT = """
function openMenu(){document.body.classList.add('menu-open')}
function closeMenu(){document.body.classList.remove('menu-open')}
function showYear(btn, y){btn.closest('[data-years]').querySelectorAll('[data-year]').forEach(d => d.hidden = d.dataset.year != y)}
document.addEventListener('keydown', e => { if (e.key === 'Escape') closeMenu() })
document.querySelectorAll('button').forEach(b => {
  if (!/Copy/.test(b.textContent)) return
  b.addEventListener('click', () => {
    const link = /league/i.test(b.textContent) ? new URL('index.html', location.href).href : location.href
    navigator.clipboard.writeText(link).then(() => { const t = b.textContent; b.textContent = 'Copied'; setTimeout(() => b.textContent = t, 1500) })
  })
})
"""

def page(title, desktop, phone, active=None):
    # Phone layouts were drawn 390 pixels wide; on a real phone they fill the screen
    phone = re.sub(r"width: 390px; min-height: \d+px", "width: 100%", phone)
    return f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{title} · {LEAGUE_NAME} · Wave Fantasy Report</title>
<link rel="icon" href="assets/wfr-icon-64.png">
<link rel="apple-touch-icon" href="assets/wfr-icon-180.png">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Alegreya+Sans+SC:wght@400;500;700;800&display=swap" rel="stylesheet">
<style>
body {{ margin: 0 }}
a {{ color: {NAVY} }} a:hover {{ color: #0B1A33 }}
.view-phone {{ display: none }}
@media (max-width: 760px) {{ .view-desktop {{ display: none }} .view-phone {{ display: block }} }}
html {{ -webkit-text-size-adjust: 100%; text-size-adjust: 100% }}
/* The menu slides in from the left while the page behind it dims */
.drawer {{ position: fixed; inset: 0; z-index: 50; visibility: hidden; background: rgba(5,15,35,0); transition: background 0.3s ease, visibility 0s linear 0.3s }}
.drawer nav {{ transform: translateX(-100%); transition: transform 0.3s ease }}
body.menu-open .drawer {{ visibility: visible; background: rgba(5,15,35,0.55); transition: background 0.3s ease }}
body.menu-open .drawer nav {{ transform: none }}
body.menu-open {{ overflow: hidden }}
@media (prefers-reduced-motion: reduce) {{ .drawer, .drawer nav {{ transition: none }} }}
/* First columns that stay put while a table scrolls sideways sit above the cells sliding under them */
[style*="position: sticky"] {{ z-index: 2; box-shadow: -10px 0 0 #FFFFFF }}
[hidden] {{ display: none !important }}
/* A missing headshot leaves a blank space, with no ring around it */
span:has(> [data-noface]) {{ box-shadow: none !important }}
</style>
</head>
<body>
<div class="view-desktop">{desktop}</div>
<div class="view-phone">{phone}</div>
{drawer(active)}
<script>{SCRIPT}</script>
</body>
</html>
'''

def names(html, c=None, d=None, nm=None):
    # [Team C] and [Team D] are the winner and loser on a matchup page; every [Team X] becomes the team's name
    if c:
        html = html.replace("[Team C]", f"[Team {c}]").replace("[Team D]", f"[Team {d}]")
    nm = nm or NAME
    html = re.sub(r"\[Team ([^\]<]+)\]", lambda m: nm.get(m.group(1), m.group(0)), html)
    for ab in nm:
        html = html.replace(f">Team {ab}<", f">{ab}<").replace(f"Team {ab} ·", f"{ab} ·")
    return html.replace("{LEAGUE_NAME}", LEAGUE_NAME)

WRITTEN = []
def write_page(file, title, desktop, phone, active=None, c=None, d=None, nm=None):
    html = names(page(title, desktop, phone, active), c, d, nm)
    (SITE / file).write_text(html, encoding="utf-8")
    WRITTEN.append(file)

## Home and start pages

`index.html` is the league home: this week's tiles, the matchups, the top of the standings, and luck. `start.html` is the page for someone who hasn't picked a league yet.

In [ ]:
def mrow(a,sa,b,sb):
    wa=sa>sb
    side=lambda n,s,w: f'<span style="display: flex; justify-content: space-between; gap: 8px; font-weight: {700 if w else 400}; color: {INK if w else INK2}"><span>{n}</span><span>{s:.1f}</span></span>'
    return f'<div style="display: flex; flex-direction: column; gap: 4px; padding: 10px 0; border-bottom: 1px solid {LINE2}">{side(a,sa,wa)}{side(b,sb,not wa)}</div>'

mat="".join(mrow(*m) for m in M)

strow="".join(f'<tr style="border-bottom: 1px solid {LINE2}"><td style="padding: 8px 6px; color: {INK3}">{i}</td><td style="padding: 8px 6px; font-weight: 700">{t}</td><td style="padding: 8px 6px; text-align: right">{r}</td><td style="padding: 8px 6px; text-align: right; color: {INK2}">{a}</td><td style="padding: 8px 6px; text-align: right">{pf:.1f}</td></tr>' for i,(t,r,a,pf) in enumerate(ST,1))

th=lambda t,al="right": f'<th style="padding: 8px 6px; text-align: {al}; font-size: 13px; color: {NAVY}; border-bottom: 2px solid {NAVY}">{t}</th>'

tile=lambda lab,val,sub,col=INK: f'<div style="display: flex; flex-direction: column; gap: 4px; padding: 16px 18px; background: #FFFFFF; border: 1px solid {LINE}; border-radius: 8px"><span style="font-size: 14px; color: {INK2}">{lab}</span><span style="font-size: 26px; font-weight: 800; color: {col}">{val}</span><span style="font-size: 14px; color: {INK2}">{sub}</span></div>'

def league_html():
    return f'''<div style="min-height: 100%; background: #FFFFFF; color: {INK}; font-family: {F}; font-size: 15px">
{navbar(league=True)}
<div style="background: {NAVY}; border-top: 1px solid #23395C">
<div style="max-width: 1240px; margin: 0 auto; padding: 18px 24px 4px; display: flex; justify-content: space-between; align-items: center; gap: 16px; flex-wrap: wrap">
<div style="display: flex; align-items: baseline; gap: 12px; flex-wrap: wrap"><span style="font-size: 30px; font-weight: 800; color: #FFFFFF">{LEAGUE_NAME}</span><span style="color: {WOLF}">{N} teams · {SCORING} · Through Week {WEEK}</span></div>
<div style="display: flex; gap: 10px">
<button style="font: inherit; min-height: 40px; padding: 0 16px; border-radius: 8px; border: 0; background: {WOLF}; color: {NAVY}; font-weight: 800">Copy league link</button>
<a href="start.html" style="display: inline-flex; align-items: center; min-height: 40px; padding: 0 16px; border-radius: 8px; color: #FFFFFF; text-decoration: none; border: 1px solid #3C5476">Change league</a>
</div></div>
<div style="background: #0E1F3A; border-top: 1px solid #23395C; border-bottom: 1px solid #23395C; margin-top: 14px; padding: 11px 24px; overflow: hidden" aria-label="This week, scrolling">
<div style="display: flex; align-items: baseline">{ticker}{ticker}</div></div>
{waves(60, NAVY)}</div>
<main style="max-width: 1240px; margin: 0 auto; padding: 4px 24px 56px; display: flex; flex-direction: column; gap: 18px">
<div style="display: flex; justify-content: space-between; align-items: baseline"><h1 style="margin: 0; font-size: 28px; font-weight: 800">Week {WEEK} at a glance</h1><a href="recap.html" style="font-weight: 700">Full recap →</a></div>
<div style="display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 14px">
{tile("Top score",TILES["top"][0],TILES["top"][1],GREEN)}
{tile("Lowest score",TILES["low"][0],TILES["low"][1],RED)}
{tile("Closest game",TILES["close"][0],TILES["close"][1])}
{tile("Biggest blowout",TILES["blow"][0],TILES["blow"][1])}
</div>
<div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 14px; align-items: start">
{card(h2("Matchups", f'<span style="font-size: 14px; color: {INK2}">Week {WEEK}</span>')+mat)}
{card(h2("Standings", '<a href="standings.html" style="font-size: 14px; font-weight: 700">All standings →</a>')+f'<table style="width: 100%; border-collapse: collapse; font-size: 15px"><thead><tr>{th("","left")}{th("Team","left")}{th("Record")}{th("All-play")}{th("PF")}</tr></thead><tbody>{strow}</tbody></table>')}
<div style="display: flex; flex-direction: column; gap: 14px">
{card(h2("Luck this season", '<a href="almost_points/index.html" style="font-size: 14px; font-weight: 700">Almost Points →</a>')+f"""<div style="display: flex; flex-direction: column; gap: 12px">
<div style="display: flex; justify-content: space-between; align-items: baseline"><span><span style="display: block; font-size: 13px; color: {INK2}">Unluckiest team</span><b style="font-size: 17px">[Team {unl}]</b></span><span style="font-size: 20px; font-weight: 800; color: {RED}">{tl[unl]:.1f} lost</span></div>
<div style="display: flex; justify-content: space-between; align-items: baseline"><span><span style="display: block; font-size: 13px; color: {INK2}">Least lost to luck</span><b style="font-size: 17px">[Team {lkst}]</b></span><span style="font-size: 20px; font-weight: 800; color: {GREEN}">{tl[lkst]:.1f} lost</span></div>
<p style="margin: 0; font-size: 13px; color: {INK3}">Points their starters lost to flags, drops, missed throws, replay, and goal-line stops.</p></div>""")}
{card(h2("Schedule luck")+f"""<div style="display: flex; flex-direction: column; gap: 10px">
{"".join(f'<div style="display: flex; justify-content: space-between"><span>[Team {r["t"]}] <span style="color: {INK2}">{r["w"]}-{r["l"]}, all-play {r["ap_w"]}-{r["ap_l"]}</span></span><b style="color: {GREEN if r["luck"]>0.5 else RED if r["luck"]<-0.5 else INK}">{"+" if r["luck"]>0 else "−"}{abs(r["luck"]):.1f} wins</b></div>' for r in SL)}
<p style="margin: 0; font-size: 13px; color: {INK3}">Actual wins minus the wins their all-play record works out to over {WEEK} games.</p></div>""")}
</div></div>
</main>
</div>'''

# ---------------- phone, league loaded ----------------
mtile=lambda lab,val,sub,col=INK: f'<div style="display: flex; flex-direction: column; gap: 2px; padding: 12px 14px; background: #FFFFFF; border: 1px solid {LINE}; border-radius: 8px"><span style="font-size: 13px; color: {INK2}">{lab}</span><span style="font-size: 22px; font-weight: 800; color: {col}">{val}</span><span style="font-size: 13px; color: {INK2}">{sub}</span></div>'

def plm_html():
    return f'''<div style="width: 390px; min-height: 1650px; background: #FFFFFF; color: {INK}; font-family: {F}; font-size: 15px">
{navbar(league=True, mobile=True)}
<div style="background: {NAVY}; border-top: 1px solid #23395C"><div style="padding: 14px 16px 2px; display: flex; flex-direction: column; gap: 10px">
<div><span style="font-size: 24px; font-weight: 800; color: #FFFFFF">{LEAGUE_NAME}</span><div style="color: {WOLF}; font-size: 14px">{N} teams · {SCORING} · Through Week {WEEK}</div></div>
<div style="display: flex; gap: 8px"><button style="flex: 1; font: inherit; min-height: 44px; border-radius: 8px; border: 0; background: {WOLF}; color: {NAVY}; font-weight: 800">Copy league link</button>
<a href="start.html" style="flex: 1; display: flex; justify-content: center; align-items: center; min-height: 44px; border-radius: 8px; border: 1px solid #3C5476; color: #FFFFFF; text-decoration: none">Change league</a></div></div>
<div style="background: #0E1F3A; border-top: 1px solid #23395C; border-bottom: 1px solid #23395C; margin-top: 12px; padding: 10px 16px; overflow: hidden"><div style="display: flex; align-items: baseline">{ticker}</div></div>
{waves(44, NAVY)}</div>
<main style="padding: 18px 16px 36px; display: flex; flex-direction: column; gap: 14px">
<div style="display: flex; justify-content: space-between; align-items: baseline"><h1 style="margin: 0; font-size: 24px; font-weight: 800">Week {WEEK}</h1><a href="recap.html" style="font-weight: 700">Full recap →</a></div>
<div style="display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 10px">
{mtile("Top score",TILES["top"][0],TILES["top"][2],GREEN)}{mtile("Lowest score",TILES["low"][0],TILES["low"][2],RED)}
{mtile("Closest game",TILES["close"][2],TILES["close"][3])}{mtile("Biggest blowout",TILES["blow"][2],TILES["blow"][3])}
</div>
{card(h2("Matchups")+"".join(mrow(*m) for m in M[:4])+f'<a href="recap.html" style="display: block; padding-top: 10px; font-weight: 700">All {len(M)} matchups →</a>',"; padding: 14px 16px")}
{card(h2("Standings", '<a href="standings.html" style="font-size: 14px; font-weight: 700">All →</a>')+f'<table style="width: 100%; border-collapse: collapse; font-size: 15px"><thead><tr>{th("","left")}{th("Team","left")}{th("Record")}{th("All-play")}</tr></thead><tbody>'+"".join(f'<tr style="border-bottom: 1px solid {LINE2}"><td style="padding: 8px 6px; color: {INK3}">{i}</td><td style="padding: 8px 6px; font-weight: 700">{t}</td><td style="padding: 8px 6px; text-align: right">{r}</td><td style="padding: 8px 6px; text-align: right; color: {INK2}">{a}</td></tr>' for i,(t,r,a,pf) in enumerate(ST,1))+'</tbody></table>',"; padding: 14px 16px")}
{card(h2("Luck this season")+f"""<div style="display: flex; justify-content: space-between; padding-bottom: 8px"><span>Unluckiest: <b>[Team {unl}]</b></span><b style="color: {RED}">{tl[unl]:.1f} lost</b></div>
<div style="display: flex; justify-content: space-between"><span>Least lost: <b>[Team {lkst}]</b></span><b style="color: {GREEN}">{tl[lkst]:.1f} lost</b></div>""","; padding: 14px 16px")}
</main></div>'''

write_page("index.html", "Home", league_html(), plm_html())
write_page("start.html", "Start", main_html(), pm_html())
print("Home pages written")

## Recap

Every week's tiles, matchups, awards, and ten best players, for every season. The pickers at the top jump between seasons and weeks, and each past season uses that season's team names. The loop ends on the latest week, so the numbers loaded afterward are this week's.

In [ ]:
def footer(mobile=False):
    if mobile:
        return f'''<footer style="background: {NAVY}; color: #FFFFFF">{waves(60, "#FFFFFF", ("#A5ACAF", "#3C5476", NAVY))}
<div style="padding: 6px 16px 28px; display: flex; flex-direction: column; gap: 6px"><img src="assets/wfr-lockup.png" alt="Wave Fantasy Report" style="height: 69px; width: auto; display: block"><span style="font-size: 15px; font-weight: 700; margin-top: 6px">New numbers every Wednesday.</span></div></footer>'''
    return f'''<footer style="background: {NAVY}; color: #FFFFFF">{waves(100, "#FFFFFF", ("#A5ACAF", "#3C5476", NAVY))}
<div style="max-width: 1240px; margin: 0 auto; padding: 6px 24px 36px; display: flex; justify-content: space-between; align-items: flex-end; gap: 24px; flex-wrap: wrap">
<div style="display: flex; flex-direction: column; gap: 6px"><img src="assets/wfr-lockup.png" alt="Wave Fantasy Report" style="height: 84px; width: auto; display: block"></div>
<span style="font-size: 17px; font-weight: 700">New numbers every Wednesday.</span></div></footer>'''

def face(size, ring=False, name=None):
    # A player's ESPN headshot, or his team's logo for a defense. A player we don't have an ID for, or a photo
    # that won't load, leaves a blank space the same size, so a broken link never shows as a stand-in picture.
    key = HEADS.get(name)
    if not key:
        return f'<span data-noface style="display: inline-block; width: {size}px; height: {size}px; flex: none"></span>'
    team = key.startswith("team:")
    url = (f"https://a.espncdn.com/i/teamlogos/nfl/500/{key[5:]}.png" if team
           else f"https://a.espncdn.com/i/headshots/nfl/players/full/{key}.png")
    b = f"border: 2px solid {WOLF}; " if ring else ""
    fit = "object-fit: contain; padding: 3px; box-sizing: border-box" if team else "object-fit: cover; object-position: 50% 15%"
    return (f'<img src="{url}" alt="" width="{size}" height="{size}" loading="lazy" referrerpolicy="no-referrer" '
            f'onerror="this.style.visibility=\'hidden\'; if (this.parentNode.style.boxShadow) this.parentNode.style.boxShadow=\'none\'" '
            f'style="{b}width: {size}px; height: {size}px; flex: none; border-radius: 50%; background: {WOLF_L}; {fit}">')

def side(t, pts, won, mobile=False):
    w = f' <span style="font-size: 12px; font-weight: 800; color: #FFFFFF; background: {GREEN}; padding: 1px 6px; border-radius: 3px; vertical-align: 2px">W</span>' if won else ''
    plays = "".join(f'<div style="display: flex; justify-content: space-between; font-size: 14px; color: {INK2}"><span>{n} <span style="color: {INK3}">{p}</span></span><span style="font-weight: 700; color: {INK}">{x:.1f}</span></div>' for n,p,x in TOP3[t])
    return f"""<div style="display: flex; flex-direction: column; gap: 5px; padding: 14px 16px">
<div style="display: flex; justify-content: space-between; align-items: baseline; gap: 10px; margin-bottom: 2px">
<span style="font-size: {19 if not mobile else 17}px; font-weight: 800; color: {NAVY}">[Team {t}]{w}</span>
<span style="font-size: {30 if not mobile else 26}px; font-weight: 800; color: {GREEN if won else INK2}">{pts:.1f}</span></div>{plays}</div>"""

def game(a,sa,b,sb, mobile=False):
    wa = sa>sb
    href = matchup_file(a, b, RW, RS)
    return f"""<a href="{href}" style="background: #FFFFFF; border: 1px solid {LINE}; border-radius: 10px; overflow: hidden; display: flex; flex-direction: column; text-decoration: none; color: {INK}">
{side(a,sa,wa,mobile)}<div style="height: 1px; background: {LINE2}; margin: 0 16px"></div>{side(b,sb,not wa,mobile)}
<div style="display: flex; justify-content: space-between; padding: 10px 16px; background: #F5F6F6; font-size: 14px"><span style="color: {INK2}">Won by {abs(sa-sb):.1f}</span><span style="font-weight: 800; color: {NAVY}">Open matchup →</span></div></a>"""

def award(t,who,d,good=True,mobile=False):
    bg=NAVY if good else WOLF_L; fg="#FFFFFF" if good else NAVY
    return f'''<div style="display: flex; flex-direction: column; gap: 4px; padding: {"14px 16px" if mobile else "18px 20px"}; background: {bg}; color: {fg}; border-radius: 12px">
<span style="font-size: 12px; font-weight: 800; letter-spacing: 1px; opacity: 0.8">{t.upper()}</span>
<span style="font-size: {24 if not mobile else 20}px; font-weight: 800; line-height: 1.1; margin-top: 2px">{who}</span><span style="font-size: 15px; opacity: 0.9">{d}</span></div>'''

def toprow(i,r,mobile=False):
    n,pos,tm,pts,lost,t,st=r
    slot = "Started" if st else f'<span style="font-weight: 800; color: {NAVY}; background: {WOLF_L}; padding: 1px 7px; border-radius: 3px">Bench</span>'
    if mobile:
        return f'''<div style="display: flex; align-items: center; gap: 10px; padding: 10px 14px; border-bottom: 1px solid {LINE2}"><span style="width: 20px; color: {INK3}; font-weight: 700">{i}</span>{face(36, name=n)}
<span style="flex: 1; min-width: 0; display: flex; flex-direction: column"><b style="font-size: 16px">{n}</b><span style="font-size: 13px; color: {INK2}">{pos} · {tm} · [Team {t}]{"" if st else " · Bench"}</span></span>
<span style="display: flex; flex-direction: column; align-items: flex-end"><b style="font-size: 18px; color: {GREEN}">{pts:.1f}</b>{f'<span style="font-size: 12px; color: {RED if lost else INK3}">{f"{lost:.1f} lost" if lost else "none lost"}</span>' if HAS_LUCK else ""}</span></div>'''
    td=lambda x,al="right",extra="": f'<td style="padding: 9px 10px; text-align: {al}; white-space: nowrap{extra}">{x}</td>'
    return (f'<tr style="border-bottom: 1px solid {LINE2}">'+td(f'<span style="color: {INK3}; font-weight: 700">{i}</span>',"left")
            +td(f'<span style="display: inline-flex; align-items: center; gap: 10px">{face(32, name=n)}<b>{n}</b><span style="color: {INK2}">{tm}</span></span>',"left")
            +td(pos,"left")+td(f'[Team {t}]',"left")+td(slot,"left")+td(f'<b style="color: {GREEN}">{pts:.1f}</b>')
            +(td(f'<span style="color: {RED if lost else "#C5CAD1"}; font-weight: 700">{f"{lost:.1f}" if lost else "—"}</span>') if HAS_LUCK else "")+'</tr>')

th=lambda t,al="right": f'<th style="padding: 10px 10px; text-align: {al}; font-size: 13px; color: {NAVY}; border-bottom: 2px solid {NAVY}; white-space: nowrap">{t}</th>'

_sel=lambda lab, opts, mobile: f'<select aria-label="{lab}" onchange="location.href=this.value" style="font: inherit; font-size: {16 if mobile else 15}px; padding: 9px 12px; border: 0; border-radius: 8px; background: #FFFFFF; color: {INK}">{opts}</select>'

weeksel=lambda mobile=False: _sel("Season", season_options(), mobile) + _sel("Week", week_options(), mobile)

tile_=lambda lab,val,sub,col=INK: f'<div style="display: flex; flex-direction: column; gap: 4px; padding: 16px 18px; background: {NAVY}; color: #FFFFFF; border-radius: 12px"><span style="font-size: 14px; font-weight: 700; opacity: 0.85">{lab}</span><span style="font-size: 30px; font-weight: 800; line-height: 1.1">{val}</span><span style="font-size: 15px; opacity: 0.9">{sub}</span></div>'

def recap_html():
    return f'''<div style="min-height: 100%; background: #FFFFFF; color: {INK}; font-family: {F}; font-size: 15px">
{navbar(active="Recap", league=True)}
<section style="position: relative; overflow: hidden; background: {NAVY}; color: #FFFFFF">
<span aria-hidden="true" style="position: absolute; right: -24px; top: -56px; font-size: 300px; font-weight: 800; line-height: 1; color: rgba(165,172,175,0.08)">WFR</span>
<div style="position: relative; max-width: 1240px; margin: 0 auto; padding: 36px 24px 14px; display: flex; justify-content: space-between; align-items: flex-end; gap: 20px; flex-wrap: wrap">
<div style="display: flex; flex-direction: column; gap: 8px"><span style="font-size: 15px; font-weight: 700; color: {WOLF}; letter-spacing: 1px">The week, settled</span>
<h1 style="margin: 0; font-size: 56px; line-height: 0.95; font-weight: 800">Week {RW} Recap</h1>
<p style="margin: 0; font-size: 18px; color: #D5D9DB">{LEAGUE_NAME} · {RS} season</p></div>
<div style="display: flex; gap: 10px; align-items: center">{weeksel()}<button style="font: inherit; min-height: 42px; padding: 0 16px; border-radius: 8px; border: 0; background: {WOLF}; color: {NAVY}; font-weight: 800">Copy link to this recap</button></div>
</div>
{waves(80, NAVY)}
</section>
<main style="max-width: 1240px; margin: 0 auto; padding: 4px 24px 44px; display: flex; flex-direction: column; gap: 30px">
<div style="display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 14px">
{tile_("Top score",TILES["top"][0],TILES["top"][1])}{tile_("Lowest score",TILES["low"][0],TILES["low"][1])}
{tile_("Closest game",TILES["close"][0],TILES["close"][1])}{tile_("Biggest blowout",TILES["blow"][0],TILES["blow"][1])}
</div>
<section style="display: flex; flex-direction: column; gap: 14px">
<div style="display: flex; justify-content: space-between; align-items: baseline; flex-wrap: wrap; gap: 8px"><h2 style="margin: 0; font-size: 28px; font-weight: 800; color: {NAVY}">Matchups</h2>
<span style="color: {INK2}">Open any matchup for the full breakdown</span></div>
<div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 14px">{"".join(game(*g) for g in GAMES)}</div>
</section>
<section style="display: flex; flex-direction: column; gap: 14px">
<h2 style="margin: 0; font-size: 28px; font-weight: 800; color: {NAVY}">Weekly awards</h2>
<div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 14px">{"".join(award(*a) for a in AWARDS)}</div>
</section>
<section style="display: flex; flex-direction: column; gap: 14px">
<div style="display: flex; justify-content: space-between; align-items: baseline"><h2 style="margin: 0; font-size: 28px; font-weight: 800; color: {NAVY}">Top performances</h2><span style="color: {INK2}">Every rostered player, bench included</span></div>
<div style="border: 1px solid {LINE}; border-radius: 10px; overflow: hidden"><table style="width: 100%; border-collapse: collapse; font-size: 15px">
<thead><tr>{th("","left")}{th("Player","left")}{th("Pos","left")}{th("Manager","left")}{th("Slot","left")}{th("Points")}{th("Lost to luck") if HAS_LUCK else ""}</tr></thead>
<tbody>{"".join(toprow(i,r) for i,r in enumerate(TOP,1))}</tbody></table></div>
</section>
</main>
{footer()}
</div>'''

def rm_html():
    return f'''<div style="width: 390px; min-height: 2750px; background: #FFFFFF; color: {INK}; font-family: {F}; font-size: 15px">
{navbar(active="Recap", league=True, mobile=True)}
<section style="position: relative; overflow: hidden; background: {NAVY}; color: #FFFFFF">
<span aria-hidden="true" style="position: absolute; right: -16px; top: -20px; font-size: 150px; font-weight: 800; line-height: 1; color: rgba(165,172,175,0.09)">WFR</span>
<div style="position: relative; padding: 22px 16px 10px; display: flex; flex-direction: column; gap: 10px">
<span style="font-size: 14px; font-weight: 700; color: {WOLF}">The week, settled</span>
<h1 style="margin: 0; font-size: 40px; line-height: 0.95; font-weight: 800">Week {RW} Recap</h1>
<p style="margin: 0; font-size: 15px; color: #D5D9DB">{LEAGUE_NAME} · {RS} season</p>
<div style="display: flex; gap: 8px; flex-wrap: wrap">{weeksel(True)}<button style="flex: 1; font: inherit; min-height: 44px; border-radius: 8px; border: 0; background: {WOLF}; color: {NAVY}; font-weight: 800">Copy link</button></div></div>
{waves(50, NAVY)}</section>
<main style="padding: 4px 16px 32px; display: flex; flex-direction: column; gap: 24px">
<div style="display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 10px">
{tile_("Top score",TILES["top"][0],TILES["top"][2])}{tile_("Lowest score",TILES["low"][0],TILES["low"][2])}{tile_("Closest game",TILES["close"][2],TILES["close"][3])}{tile_("Biggest blowout",TILES["blow"][2],TILES["blow"][3])}</div>
<section style="display: flex; flex-direction: column; gap: 12px"><h2 style="margin: 0; font-size: 24px; font-weight: 800; color: {NAVY}">Matchups</h2>
<span style="color: {INK2}; margin-top: -6px">Tap any matchup for the full breakdown</span>{"".join(game(*g, mobile=True) for g in GAMES)}</section>
<section style="display: flex; flex-direction: column; gap: 10px"><h2 style="margin: 0; font-size: 24px; font-weight: 800; color: {NAVY}">Weekly awards</h2>{"".join(award(*a, mobile=True) for a in AWARDS)}</section>
<section style="display: flex; flex-direction: column; gap: 10px"><h2 style="margin: 0; font-size: 24px; font-weight: 800; color: {NAVY}">Top performances</h2>
<div style="border: 1px solid {LINE}; border-radius: 10px; overflow: hidden">{"".join(toprow(k+1,TOP[k],True) for k in range(len(TOP)))}</div></section>
</main>
{footer(True)}
</div>'''

for RS, RW in sorted(RECAPS):
    globals().update({k: v for k, v in RECAPS[(RS, RW)].items() if k not in ("W", "MATCHUPS")})
    title = f"Week {RW} Recap" if RS == SEASON else f"{RS} Week {RW} Recap"
    write_page(recap_file(RS, RW), title, recap_html(), rm_html(), active="Recap", nm={**NAME, **SEASON_NAMES.get(RS, {})})
print(len(RECAPS), "weekly recaps written")

## Matchups

One page per game, every week: the story, the "If…" line, where the points went, the standouts, and both lineups. Each game's numbers come from `01_build_data` and are loaded in turn before its page is built.

In [ ]:
beat=lambda L: sum(1 for r in L if r[4]-r[5] > 0.05)

tot=lambda L: sum(r[4] for r in L); proj=lambda L: sum(r[5] for r in L); luck=lambda L: sum(r[6] for r in L)

ROW_H = 54     # every starter row is the same height so the two lineups line up

BENCH_H = 34

def lineup(team, L, B, won, pad_to, mobile=False):
    CELLPAD = "padding: 0 4px" if mobile else "padding: 0 8px"
    def row(r):
        slot,n,p,tm,pts,pj,lk,rk,td=r; d=pts-pj
        dcol = GREEN if d>0.05 else (RED if d<-0.05 else INK3)
        rank = f"{p}{rk}" if rk else "—"
        luckline = f'<span style="display: block; font-size: 12px; color: {RED}; line-height: 1.2">{lk:.1f} lost to luck</span>' if lk else ""
        return (f'<tr style="height: {ROW_H}px; border-bottom: 1px solid {LINE2}"><td style="{CELLPAD}; color: {INK3}; font-weight: 700; width: 44px">{slot}</td>'
                f'<td style="{CELLPAD}"><span style="display: inline-flex; align-items: center; gap: 8px">{face(26, name=n) if p != "K" else ""}<span><b>{n}</b>{luckline}</span></span></td>'
                f'<td style="{CELLPAD}; text-align: right; color: {INK2}; white-space: nowrap">{rank}</td>'
                f'<td style="{CELLPAD}; text-align: right; color: {INK3}">{pj:.1f}</td>'
                f'<td style="{CELLPAD}; text-align: right; font-weight: 800">{pts:.1f}</td>'
                f'<td style="{CELLPAD}; text-align: right; color: {dcol}; font-weight: 700">{"+" if d>0 else ("−" if d<0 else "")}{abs(d):.1f}</td></tr>')
    bench = "".join(f'<tr style="height: {BENCH_H}px; border-bottom: 1px solid {LINE2}; color: {INK3}; font-size: 13px"><td style="{CELLPAD}; font-weight: 700">BN</td><td style="{CELLPAD}">{n} <span>{p}</span></td><td></td><td></td><td style="{CELLPAD}; text-align: right">{x:.1f}</td><td></td></tr>' for n,p,tm,x in B)
    bench += "".join(f'<tr style="height: {BENCH_H}px"><td colspan="6"></td></tr>' for _ in range(pad_to - len(B)))
    th_=lambda t,al="right": f'<th style="padding: 8px 8px; text-align: {al}; font-size: 13px; color: {NAVY}; border-bottom: 2px solid {NAVY}">{t}</th>'
    return f"""<section style="background: #FFFFFF; border: 1px solid {LINE}; border-radius: 10px; overflow-x: auto">
<div style="display: flex; justify-content: space-between; align-items: baseline; padding: 14px 16px; background: {NAVY if won else '#F5F6F6'}; color: {'#FFFFFF' if won else INK}">
<span style="font-size: 22px; font-weight: 800">[Team {team}]{' · W' if won else ''}</span><span style="font-size: 28px; font-weight: 800">{tot(L):.1f}</span></div>
<table style="width: 100%; border-collapse: collapse; font-size: 15px"><thead><tr>{th_("","left")}{th_("Starter","left")}{th_("Week")}{th_("Proj")}{th_("Pts")}{th_("+/−")}</tr></thead>
<tbody>{"".join(row(r) for r in L)}<tr style="height: 34px"><td colspan="6" style="{CELLPAD}; font-size: 13px; font-weight: 800; color: {INK3}">Bench · {sum(b[3] for b in B):.1f} points</td></tr>{bench}</tbody></table></section>"""

sg=lambda x: ("+" if x > 0.05 else "−" if x < -0.05 else "") + f"{abs(x):.1f}"

def stat(label, c, d, note="", worse="high", fmt_=lambda x: f"{x:.1f}"):
    # worse = "high" shades the bigger number red, "low" the smaller one, None shades nothing
    if worse is None or abs(c-d) < 0.05: cc = dc = INK
    else:
        c_bad = c > d if worse == "high" else c < d
        cc, dc = (RED, INK) if c_bad else (INK, RED)
    return f"""<div style="display: grid; grid-template-columns: minmax(0, 1fr) auto minmax(0, 1fr); align-items: center; gap: 14px; padding: 12px 0; border-bottom: 1px solid {LINE2}">
<span style="text-align: right; font-size: 24px; font-weight: 800; color: {cc}">{fmt_(c)}</span>
<span style="text-align: center; font-size: 15px; font-weight: 700; color: {NAVY}; max-width: 300px">{label}{f'<span style="display: block; font-size: 13px; font-weight: 400; color: {INK2}">{note}</span>' if note else ''}</span>
<span style="font-size: 24px; font-weight: 800; color: {dc}">{fmt_(d)}</span></div>"""

SO_ORDER=["Stud","Dud","Best start","Worst start","Highest scorer","Lowest scorer"]               # phone

ORDER_DESKTOP=["Stud","Best start","Highest scorer","Dud","Worst start","Lowest scorer"]  # good row over bad row

def standout_card(t, mobile=False):
    def half(team, S):
        who, d = S[t]
        d = re.sub(r"(?<![A-Za-z])-(?=\d)", "−", d)
        return f'<div style="display: flex; flex-direction: column; gap: 2px; padding-top: 8px"><span style="font-size: 13px; font-weight: 700; opacity: 0.7">[Team {team}]</span><span style="font-size: 18px; font-weight: 800">{who}</span><span style="font-size: 14px; opacity: 0.85; line-height: 1.35">{d}</span></div>'
    good=t in ("Stud","Best start","Highest scorer")
    return f'''<div style="display: flex; flex-direction: column; gap: 4px; padding: {"14px 16px" if mobile else "16px 18px"}; background: {NAVY if good else WOLF_L}; color: {"#FFFFFF" if good else NAVY}; border-radius: 12px">
<span style="font-size: 12px; font-weight: 800; letter-spacing: 1px; opacity: 0.8">{t.upper()}</span>
<div style="display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 14px">{half("C", SC_)}{half("D", SD)}</div></div>'''

def detail(mobile=False):
    pad = "16px" if mobile else "24px"
    story = f'''<section style="background: #F5F6F6; border-radius: 10px; padding: {pad}; display: flex; flex-direction: column; gap: 8px">
<span style="font-size: 12px; font-weight: 800; letter-spacing: 1px; color: #FFFFFF; background: {NAVY}; padding: 2px 8px; border-radius: 3px; align-self: flex-start">The story</span>
<p style="margin: 0; font-size: {16 if mobile else 18}px; line-height: 1.5">{STORY}</p></section>'''
    iff = f'''<section style="border: 2px solid {NAVY}; border-radius: 10px; padding: {"14px 16px" if mobile else "18px 22px"}; display: flex; gap: 14px; align-items: flex-start">
<span style="flex: none; font-size: 30px; font-weight: 800; color: {NAVY}; line-height: 1">If…</span><p style="margin: 0; font-size: {16 if mobile else 18}px; line-height: 1.45">{IFLINE}</p></section>'''
    diff = f'''<section style="background: #FFFFFF; border: 1px solid {LINE}; border-radius: 10px; padding: {"6px 16px 10px" if mobile else "8px 24px 12px"}">
<div style="display: grid; grid-template-columns: minmax(0, 1fr) auto minmax(0, 1fr); gap: 14px; padding: 10px 0; border-bottom: 2px solid {NAVY}; font-weight: 800; color: {NAVY}"><span style="text-align: right">[Team C]</span><span>Where it went</span><span>[Team D]</span></div>
{stat("Projected", proj(LC), proj(LD), worse=None)}
{stat("Lineup score", EFF_C, EFF_D, worse="low", fmt_=lambda x: f"{x:.1f}%")}
{stat("Lost to luck", luck(LC), luck(LD)) if HAS_LUCK else ""}
{stat("Left on the bench", sum(b[3] for b in BC), sum(b[3] for b in BD))}
{stat("Vs. league median", tot(LC)-MEDIAN, tot(LD)-MEDIAN, worse=None, fmt_=sg)}
{stat("Vs. season average", tot(LC)-SEASON_AVG["C"], tot(LD)-SEASON_AVG["D"], worse=None, fmt_=sg)}
</section>'''
    pad_to = max(len(BC), len(BD))
    lineups = (f'<div style="display: flex; flex-direction: column; gap: 14px">{lineup("C",LC,BC,True,len(BC),True)}{lineup("D",LD,BD,False,len(BD),True)}</div>' if mobile else
               f'<div style="display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 14px; align-items: start">{lineup("C",LC,BC,True,pad_to)}{lineup("D",LD,BD,False,pad_to)}</div>')
    cards = "".join(standout_card(t, mobile) for t in (SO_ORDER if mobile else ORDER_DESKTOP))
    grid = f'<div style="display: grid; grid-template-columns: repeat({1 if mobile else 3}, minmax(0, 1fr)); gap: {10 if mobile else 14}px">{cards}</div>'
    h2 = lambda t: f'<h2 style="margin: 0; font-size: {24 if mobile else 28}px; font-weight: 800; color: {NAVY}">{t}</h2>'
    back = recap_file(RS, RW)
    head = f'''<section style="position: relative; overflow: hidden; background: {NAVY}; color: #FFFFFF">
<span aria-hidden="true" style="position: absolute; right: {-16 if mobile else -24}px; top: {-20 if mobile else -56}px; font-size: {150 if mobile else 300}px; font-weight: 800; line-height: 1; color: rgba(165,172,175,0.08)">WFR</span>
<div style="position: relative; {"padding: 14px 16px 10px" if mobile else "max-width: 1240px; margin: 0 auto; padding: 20px 24px 14px"}; display: flex; flex-direction: column; gap: 14px">
<a href="{back}" style="align-self: flex-start; display: inline-flex; align-items: center; min-height: {44 if mobile else 38}px; padding: 0 14px; border-radius: 8px; background: {WOLF}; color: {NAVY}; text-decoration: none; font-weight: 800">← Week {RW} Recap</a>
<div style="display: flex; align-items: center; gap: {12 if mobile else 26}px; flex-wrap: wrap">
<span style="display: flex; flex-direction: column"><span style="font-size: {16 if mobile else 20}px; color: {WOLF}">[Team C] · W</span><span style="font-size: {48 if mobile else 72}px; font-weight: 800; line-height: 0.95">{tot(LC):.1f}</span></span>
<span style="font-size: {22 if mobile else 30}px; color: {WOLF}; font-weight: 800">to</span>
<span style="display: flex; flex-direction: column"><span style="font-size: {16 if mobile else 20}px; color: {WOLF}">[Team D]</span><span style="font-size: {48 if mobile else 72}px; font-weight: 800; line-height: 0.95; color: #D5D9DB">{tot(LD):.1f}</span></span>
</div><span style="font-size: {15 if mobile else 18}px; color: #D5D9DB">{RS} · Week {RW} · Won by {tot(LC)-tot(LD):.1f}</span></div>
{waves(50 if mobile else 80, NAVY)}</section>'''
    body = f'''<main style="{"padding: 4px 16px 32px" if mobile else "max-width: 1240px; margin: 0 auto; padding: 4px 24px 44px"}; display: flex; flex-direction: column; gap: {22 if mobile else 26}px">
{story}{iff}{diff}{h2("Standouts")}{grid}{h2("Lineups")}{lineups}</main>'''
    root_style = f'width: 390px; min-height: 4250px' if mobile else 'min-height: 100%'
    return f'''<div style="{root_style}; background: #FFFFFF; color: {INK}; font-family: {F}; font-size: 15px">
{navbar(active="Recap", league=True, mobile=mobile)}{head}{body}{footer(mobile)}</div>'''

n = 0
for RS, RW in sorted(RECAPS):
    for m in RECAPS[(RS, RW)]["MATCHUPS"]:
        globals().update(m)
        write_page(matchup_file(MATCH_A, MATCH_B, RW, RS), f"{RS} Week {RW}: [Team {MATCH_A}] vs. [Team {MATCH_B}]", detail(), detail(True),
                   active="Recap", c=MATCH_A, d=MATCH_B, nm={**NAME, **SEASON_NAMES.get(RS, {})})
        n += 1
RS, RW = SEASON, WEEK
globals().update({k: v for k, v in RECAPS[(SEASON, WEEK)].items() if k not in ("W", "MATCHUPS")})
print(n, "matchup pages written")

## Standings

The standings table, schedule luck, the schedule swap grid, week by week, and points by position.

In [ ]:
sgn=lambda x: ("+" if x>0.05 else "−" if x<-0.05 else "")+f"{abs(x):.1f}"

def st_table():
    th_=lambda t,al="right",extra="": f'<th style="padding: 10px 10px; text-align: {al}; font-size: 13px; color: {NAVY}; border-bottom: 2px solid {NAVY}; white-space: nowrap{extra}">{t}</th>'
    SEPX="; border-left: 1px solid #EEF0F2"
    head = th_("","left")+th_("Team","left")+th_("Record")+th_("PF")+th_("PA")+th_("Lineup score")+th_("All-play",extra=SEPX)+th_("Vs. median")+th_("Expected wins")+th_("Schedule luck")+th_("",extra=SEPX)
    body=""
    for r in rows:
        td=lambda x,al="right",extra="": f'<td style="padding: 11px 10px; text-align: {al}; white-space: nowrap{extra}">{x}</td>'
        lk=r["luck"]; lcol = GREEN if lk>0.25 else (RED if lk<-0.25 else INK2)
        stt = {"Clinched": f'<span style="font-size: 12px; font-weight: 800; color: #FFFFFF; background: {GREEN}; padding: 2px 8px; border-radius: 3px">Clinched</span>',
               "Eliminated": f'<span style="font-size: 12px; font-weight: 800; color: {INK2}; background: {WOLF_L}; padding: 2px 8px; border-radius: 3px">Eliminated</span>'}.get(r["status"],"")
        line = f"; border-bottom: 3px solid {NAVY}" if r["rank"]==PLAYOFF_TEAMS else f"; border-bottom: 1px solid {LINE2}"
        body+=(f'<tr style="{line[2:]}">'+td(f'<span style="color: {INK3}; font-weight: 700">{r["rank"]}</span>',"left")
               +td(f'<b style="font-size: 16px; color: {NAVY}">[Team {r["t"]}]</b>',"left")
               +td(f'<b style="font-size: 16px">{r["w"]}-{r["l"]}</b>')+td(f'{r["pf"]:.1f}')+td(f'<span style="color: {INK2}">{r["pa"]:.1f}</span>')+td(f'{r["lineup"]:.1f}%')
               +td(f'{r["ap_w"]}-{r["ap_l"]}',extra=SEPX)+td(f'{r["mw"]}-{r["ml"]}')+td(f'{r["exp"]:.1f}')
               +td(f'<b style="color: {lcol}">{sgn(lk)}</b>')+td(stt,"left",SEPX)+'</tr>')
    return f'<div style="border: 1px solid {LINE}; border-radius: 10px; overflow: hidden"><table style="width: 100%; border-collapse: collapse; font-size: 15px"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'

def swap_grid(mobile=False):
    order=[r["t"] for r in rows]
    cell=lambda t,s: SWAP[t][s]
    head='<th style="padding: 8px; position: sticky; left: 0; background: #FFFFFF"></th>'+"".join(f'<th style="padding: 8px 6px; font-size: 13px; color: {NAVY}; border-bottom: 2px solid {NAVY}; white-space: nowrap">{s}</th>' for s in order)
    body=""
    for t in order:
        own=SWAP[t][t][0]; cells=""
        for s in order:
            w,l=cell(t,s); dlt=w-own
            if s==t: bg=NAVY; col="#FFFFFF"; fw=800
            elif dlt>=2: bg="#CDEBD8"; col=INK; fw=700
            elif dlt==1: bg="#E8F5ED"; col=INK; fw=500
            elif dlt<=-2: bg="#F6CFCD"; col=INK; fw=700
            elif dlt==-1: bg="#FBEAEA"; col=INK; fw=500
            else: bg="#FFFFFF"; col=INK2; fw=500
            cells+=f'<td style="padding: 0; text-align: center"><span style="display: block; margin: 2px; padding: 8px 4px; border-radius: 4px; background: {bg}; color: {col}; font-weight: {fw}; white-space: nowrap">{w}-{l}</span></td>'
        body+=f'<tr><th style="padding: 6px 10px; text-align: left; font-size: 14px; color: {NAVY}; white-space: nowrap; position: sticky; left: 0; background: #FFFFFF">[Team {t}]</th>{cells}</tr>'
    return f'''<div style="border: 1px solid {LINE}; border-radius: 10px; overflow-x: auto; padding: 8px"><table style="border-collapse: collapse; font-size: 14px; width: 100%; min-width: {760 if mobile else 0}px"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'''

def st_cards():
    out=""
    for r in rows:
        lk=r["luck"]; lcol = GREEN if lk>0.25 else (RED if lk<-0.25 else INK2)
        stt = f' · <span style="color: {GREEN if r["status"]=="Clinched" else INK3}; font-weight: 700">{r["status"]}</span>' if r["status"] else ""
        line = f"border-bottom: 3px solid {NAVY}" if r["rank"]==PLAYOFF_TEAMS else f"border-bottom: 1px solid {LINE2}"
        out+=f'''<div style="display: flex; align-items: center; gap: 12px; padding: 12px 14px; {line}">
<span style="width: 22px; color: {INK3}; font-weight: 700">{r["rank"]}</span>
<span style="flex: 1; min-width: 0; display: flex; flex-direction: column; gap: 2px"><b style="font-size: 17px; color: {NAVY}">[Team {r["t"]}]</b>
<span style="font-size: 13px; color: {INK2}">All-play {r["ap_w"]}-{r["ap_l"]} · vs. median {r["mw"]}-{r["ml"]} · lineup {r["lineup"]:.1f}%{stt}</span></span>
<span style="display: flex; flex-direction: column; align-items: flex-end"><b style="font-size: 19px">{r["w"]}-{r["l"]}</b><span style="font-size: 13px; color: {lcol}; font-weight: 700">{sgn(lk)} wins</span></span></div>'''
    return f'<div style="border: 1px solid {LINE}; border-radius: 10px; overflow: hidden">{out}</div>'

def band(title, kicker, sub, mobile=False):
    return f'''<section style="position: relative; overflow: hidden; background: {NAVY}; color: #FFFFFF">
<span aria-hidden="true" style="position: absolute; right: {-16 if mobile else -24}px; top: {-20 if mobile else -56}px; font-size: {150 if mobile else 300}px; font-weight: 800; line-height: 1; color: rgba(165,172,175,0.08)">WFR</span>
<div style="position: relative; {"padding: 18px 16px 10px" if mobile else "max-width: 1240px; margin: 0 auto; padding: 24px 24px 14px"}; display: flex; flex-direction: column; gap: 8px">
<span style="font-size: {14 if mobile else 15}px; font-weight: 700; color: {WOLF}; letter-spacing: 1px">{kicker}</span>
<h1 style="margin: 0; font-size: {40 if mobile else 56}px; line-height: 0.95; font-weight: 800">{title}</h1>
<p style="margin: 0; font-size: {15 if mobile else 18}px; color: #D5D9DB">{sub}</p></div>
{waves(50 if mobile else 80, NAVY)}</section>'''

best=max(rows,key=lambda r:r["luck"]); worst=min(rows,key=lambda r:r["luck"])

# A team needs a real sample of wins (or losses) to be in the running: at least a quarter of the
# games played, and never fewer than 3. With 11 games that's 3.
MIN_GAMES=max(3, math.ceil(WEEKS*0.25))

def avg_margin(t, wins):
    ms=[abs(SC[t][k]-SC[opp[t][k]][k]) for k in range(WEEKS) if (SC[t][k]>SC[opp[t][k]][k])==wins]
    return sum(ms)/len(ms) if len(ms) >= MIN_GAMES else None

heart=min((r for r in rows if avg_margin(r["t"],False) is not None), key=lambda r: avg_margin(r["t"],False))

roller=max((r for r in rows if avg_margin(r["t"],True) is not None), key=lambda r: avg_margin(r["t"],True))

def lucktiles(mobile=False):
    t=lambda lab,who,val,col=INK: f'<div style="display: flex; flex-direction: column; gap: 4px; padding: 16px 18px; background: #FFFFFF; border: 1px solid {LINE}; border-radius: 10px"><span style="font-size: 14px; color: {INK2}">{lab}</span><span style="font-size: {22 if mobile else 26}px; font-weight: 800; color: {NAVY}">[Team {who}]</span><span style="font-size: 15px; color: {col}; font-weight: 700">{val}</span></div>'
    return (f'<div style="display: grid; grid-template-columns: repeat({2 if mobile else 4}, minmax(0, 1fr)); gap: {10 if mobile else 14}px">'
            + t("Luckiest schedule",best["t"],f'{sgn(best["luck"])} wins over expected',GREEN)
            + t("Unluckiest schedule",worst["t"],f'{sgn(worst["luck"])} wins under expected',RED)
            + t("Heartbreaker",heart["t"],f'Loses by just {avg_margin(heart["t"],False):.1f} on average')
            + t("Steamroller",roller["t"],f'Wins by {avg_margin(roller["t"],True):.1f} on average') + '</div>')

h2=lambda t,right="",mobile=False: f'<div style="display: flex; justify-content: space-between; align-items: baseline; gap: 10px; flex-wrap: wrap"><h2 style="margin: 0; font-size: {24 if mobile else 28}px; font-weight: 800; color: {NAVY}">{t}</h2><span style="color: {INK2}">{right}</span></div>'

KEY = f'<span style="color: {INK2}">Navy is each team&#39;s real record. Green is a better record with that schedule, red a worse one.</span>'

wk_rank={t:[sorted(TEAMS,key=lambda x:-SC[x][k]).index(t)+1 for k in range(WEEKS)] for t in TEAMS}

won=lambda t,k: SC[t][k]>SC[opp[t][k]][k]

REMAIN=[]   # future matchups not in this pull

ropp={t:[] for t in TEAMS}

for rd in REMAIN:
    for a,b in rd: ropp[a].append(b); ropp[b].append(a)

ppg=lambda t: sum(SC[t])/WEEKS

in_cut=[r["t"] for r in rows[:PLAYOFF_TEAMS]]

cut_w=rows[PLAYOFF_TEAMS-1]["w"]

EX={}

for r in rows:
    t=r["t"]
    margins=[SC[t][k]-SC[opp[t][k]][k] for k in range(WEEKS)]
    close=[m for m in margins if abs(m)<10]
    # streaks
    seq=["W" if won(t,k) else "L" for k in range(WEEKS)]
    cur=1
    for k in range(WEEKS-2,-1,-1):
        if seq[k]==seq[-1]: cur+=1
        else: break
    def longest(ch):
        best=run=0
        for x in seq:
            run = run+1 if x==ch else 0; best=max(best,run)
        return best
    vp=[k for k in range(WEEKS) if opp[t][k] in in_cut and opp[t][k]!=t]
    EX[t]=dict(close_w=sum(1 for m in close if m>0), close_l=sum(1 for m in close if m<0),
               streak=f"{seq[-1]}{cur}", lw=longest("W"), ll=longest("L"),
               gb=(cut_w - r["w"]) if r["rank"]>PLAYOFF_TEAMS else None,
               rem=(sum(ppg(o) for o in ropp[t])/len(ropp[t])) if ropp[t] else 0,
               tops=sum(1 for k in range(WEEKS) if wk_rank[t][k]==1), bots=sum(1 for k in range(WEEKS) if wk_rank[t][k]==12),
               vp_w=sum(1 for k in vp if won(t,k)), vp_l=sum(1 for k in vp if not won(t,k)),
               best=max(SC[t]), worst=min(SC[t]), sd=statistics.pstdev(SC[t]))

def sec(title, inner, mobile=False, note=""):
    return f'''<section style="display: flex; flex-direction: column; gap: 12px">{h2(title, note, mobile)}{inner}</section>'''

# bright, shiny gold: a light-to-deep sweep with a highlight band, a darker rim, and a soft glow
GOLD="linear-gradient(140deg, #FFF6C2 0%, #FFD93D 30%, #FFC300 55%, #FFE680 78%, #F2B100 100%)"

GOLD_EDGE="inset 0 0 0 2px #D19A00, inset 0 1px 0 2px rgba(255,255,255,0.65), 0 2px 10px rgba(255,195,0,0.55)"

def heatmap(mobile=False):
    def col(rk):
        # 1 = best score in the league that week (deep green) to 12 = worst (deep red)
        scale=["#1F8A4C","#3E9E63","#6BB487","#9ACBAD","#C7E2D1","#EEF0F2","#EEF0F2","#F6D2D0","#F0A9A5","#E87C77","#D9534E","#C9302C"]
        return scale[rk-1]
    th_=lambda t, extra="": f'<th style="padding: 6px 4px; font-size: 13px; color: {NAVY}; border-bottom: 2px solid {NAVY}; white-space: nowrap{extra}">{t}</th>'
    head=f'<th style="padding: 6px 8px; position: sticky; left: 0; background: #FFFFFF; border-bottom: 2px solid {NAVY}"></th>'+"".join(th_(f"Wk {k+1}") for k in range(WEEKS))+th_("Avg", f"; border-left: 2px solid {NAVY}")
    body=""
    for r in rows:
        t=r["t"]; cells=""
        for k in range(WEEKS):
            rk=wk_rank[t][k]; w=won(t,k)
            gold = rk==1          # the week's high score
            dark = (rk<=3 or rk>=10) and not gold
            tag = (f'<span style="display: block; margin-top: 3px; font-size: 10px; font-weight: 800; letter-spacing: 1px; '
                   f'color: {NAVY if gold else ("#FFFFFF" if dark else (INK if w else INK2))}; opacity: {0.95 if w else 0.75}">{"WIN" if w else "LOSS"}</span>')
            cells+=(f'<td style="padding: 0"><span title="Week {k+1}: {SC[t][k]:.1f}, {rk} of {N}" style="display: block; margin: 2px; padding: 6px 2px 5px; min-width: 52px; text-align: center; border-radius: 5px; '
                    f'background: {GOLD if gold else col(rk)}; color: {NAVY if gold else ("#FFFFFF" if dark else INK)}; font-weight: {800 if gold else 700}; line-height: 1{f"; box-shadow: {GOLD_EDGE}" if gold else ""}">{SC[t][k]:.1f}{tag}</span></td>')
        avg=sum(SC[t])/WEEKS
        cells+=f'<td style="padding: 0 10px; text-align: right; font-weight: 800; color: {NAVY}; border-left: 2px solid {NAVY}; white-space: nowrap">{avg:.1f}</td>'
        body+=f'<tr><th style="padding: 4px 10px; text-align: left; font-size: 14px; color: {NAVY}; white-space: nowrap; position: sticky; left: 0; background: #FFFFFF">[Team {t}]</th>{cells}</tr>'
    return f'<div style="border: 1px solid {LINE}; border-radius: 10px; overflow-x: auto; padding: 8px"><table style="border-collapse: collapse; font-size: 14px; width: 100%; min-width: 900px"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'

POS_COLORS=[("QB","#13294B"),("RB","#2F4A75"),("WR","#5A7299"),("TE","#8C9DBA"),("K","#A5ACAF"),("DEF","#C9CED1")]

def pos_chart(mobile=False):
    # same look as Week by week: each cell is shaded by its rank among the 12 teams in that column
    scale=["#1F8A4C","#3E9E63","#6BB487","#9ACBAD","#C7E2D1","#EEF0F2","#EEF0F2","#F6D2D0","#F0A9A5","#E87C77","#D9534E","#C9302C"]
    cols=[p for p,_ in POS_COLORS]
    val=lambda t,i: POS_SPLIT[t][i]
    rank={i:{t:sorted(TEAMS,key=lambda x:-val(x,i)).index(t)+1 for t in TEAMS} for i in range(len(cols))}
    th_=lambda t, extra="": f'<th style="padding: 6px 4px; font-size: 13px; color: {NAVY}; border-bottom: 2px solid {NAVY}; white-space: nowrap{extra}">{t}</th>'
    head=f'<th style="padding: 6px 8px; position: sticky; left: 0; background: #FFFFFF; border-bottom: 2px solid {NAVY}"></th>'+"".join(th_(c) for c in cols)+th_("Total", f"; border-left: 2px solid {NAVY}")
    body=""
    for r in rows:
        t=r["t"]; cells=""
        for i in range(len(cols)):
            rk=rank[i][t]; gold = rk==1          # the league's best at that position
            dark = (rk<=3 or rk>=10) and not gold
            cells+=(f'<td style="padding: 0"><span title="{cols[i]}: {val(t,i):.1f}, {rk} of {N}" style="display: block; margin: 2px; padding: 10px 2px; min-width: 64px; text-align: center; border-radius: 5px; '
                    f'background: {GOLD if gold else scale[rk-1]}; color: {NAVY if gold else ("#FFFFFF" if dark else INK)}; font-weight: {800 if gold else 700}{f"; box-shadow: {GOLD_EDGE}" if gold else ""}">{val(t,i):.1f}</span></td>')
        cells+=f'<td style="padding: 0 10px; text-align: right; font-weight: 800; color: {NAVY}; border-left: 2px solid {NAVY}; white-space: nowrap">{r["pf"]:.1f}</td>'
        body+=f'<tr><th style="padding: 4px 10px; text-align: left; font-size: 14px; color: {NAVY}; white-space: nowrap; position: sticky; left: 0; background: #FFFFFF">[Team {t}]</th>{cells}</tr>'
    return f'<div style="border: 1px solid {LINE}; border-radius: 10px; overflow-x: auto; padding: 8px"><table style="border-collapse: collapse; font-size: 14px; width: 100%; min-width: 560px"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'

SCALE=["#1F8A4C","#3E9E63","#6BB487","#9ACBAD","#C7E2D1","#EEF0F2","#EEF0F2","#F6D2D0","#F0A9A5","#E87C77","#D9534E","#C9302C"]

def grid(cells_for, cols, mobile=False, end=None, minw=52):
    # shared grid look: teams down the left in standings order, columns across the top
    th_=lambda t, extra="": f'<th style="padding: 6px 4px; font-size: 13px; color: {NAVY}; border-bottom: 2px solid {NAVY}; white-space: nowrap{extra}">{t}</th>'
    head=f'<th style="padding: 6px 8px; position: sticky; left: 0; background: #FFFFFF; border-bottom: 2px solid {NAVY}"></th>'+"".join(th_(c) for c in cols)+(th_(end[0], f"; border-left: 2px solid {NAVY}") if end else "")
    body=""
    for r in rows:
        t=r["t"]; cells=""
        for txt,bg,fg,tip,extra in cells_for(t):
            cells+=f'<td style="padding: 0"><span title="{tip}" style="display: block; margin: 2px; padding: 9px 2px; min-width: {minw}px; text-align: center; border-radius: 5px; background: {bg}; color: {fg}; font-weight: 700{extra}">{txt}</span></td>'
        if end: cells+=f'<td style="padding: 0 10px; text-align: right; font-weight: 800; color: {NAVY}; border-left: 2px solid {NAVY}; white-space: nowrap">{end[1](t)}</td>'
        body+=f'<tr><th style="padding: 4px 10px; text-align: left; font-size: 14px; color: {NAVY}; white-space: nowrap; position: sticky; left: 0; background: #FFFFFF">[Team {t}]</th>{cells}</tr>'
    return f'<div style="border: 1px solid {LINE}; border-radius: 10px; overflow-x: auto; padding: 8px"><table style="border-collapse: collapse; font-size: 14px; width: 100%; min-width: {len(cols)*(minw+4)+150}px"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'

def standing_after(k):
    rec={x:sum(1 for j in range(k+1) if won(x,j)) for x in TEAMS}
    pts={x:sum(SC[x][:k+1]) for x in TEAMS}
    order=sorted(TEAMS,key=lambda x:(-rec[x],-pts[x]))
    return {x:order.index(x)+1 for x in TEAMS}

STAND=[standing_after(k) for k in range(WEEKS)]

def extras(mobile=False):
    wk=[f"Wk {k+1}" for k in range(WEEKS)]
    return (sec("Week by week", heatmap(mobile), mobile, "Every weekly score, shaded by where it ranked in the league that week. <br>")
            + sec("Points by position", pos_chart(mobile), mobile, "Season points from each position, shaded by where it ranks in the league")
            )

def desk_html():
    return f'''<div style="min-height: 100%; background: #FFFFFF; color: {INK}; font-family: {F}; font-size: 15px">
{navbar(active="Standings", league=True)}
{band("Standings","Your record, and the one you earned",f"{LEAGUE_NAME} · Through Week {WEEK}")}
<main style="max-width: 1240px; margin: 0 auto; padding: 4px 24px 44px; display: flex; flex-direction: column; gap: 26px">
{lucktiles()}
{st_table()}
<p style="margin: -12px 0 0; font-size: 14px; color: {INK2}">The navy line marks the playoff cut.  All-play is the record from playing every team every week. Expected wins are the all-play win rate over {WEEK} games, and schedule luck is real wins minus expected wins. <br>Lineup score is the points started out of the best lineup each roster allowed, over the season.</p>
{h2("Schedule swap","Each row&#39;s record if they&#39;d had the column team&#39;s schedule")}
{swap_grid()}
{KEY}
{extras()}
</main>
{footer()}
</div>'''

def mob_html():
    return f'''<div style="width: 390px; min-height: 3900px; background: #FFFFFF; color: {INK}; font-family: {F}; font-size: 15px">
{navbar(active="Standings", league=True, mobile=True)}
{band("Standings","Your record, and the one you earned",f"{LEAGUE_NAME} · Through Week {WEEK}", True)}
<main style="padding: 4px 16px 32px; display: flex; flex-direction: column; gap: 20px">
{lucktiles(True)}
{st_cards()}
<p style="margin: -8px 0 0; font-size: 13px; color: {INK2}">The navy line marks the playoff cut, set by your league&#39;s playoff spots. The green and red number is schedule luck: real wins minus the wins the all-play record works out to.</p>
{h2("Schedule swap","",True)}<span style="color: {INK2}; margin-top: -12px">Each row&#39;s record with the column team&#39;s schedule. Swipe sideways.</span>
{swap_grid(True)}
{extras(True)}
</main>
{footer(True)}
</div>'''

write_page("standings.html", "Standings", desk_html(), mob_html(), active="Standings")
print("Standings written")

## Teams

One page per team, with the team switcher linking all of them. Each team's numbers are loaded in turn before its page is built.

In [ ]:
POOL=sorted(v for t in TEAMS for v in LUCKW[t])

pct_worse=lambda v: 100*sum(1 for x in POOL if x < v)/len(POOL)     # share of team-weeks with less lost

season_avg=lambda t: sum(LUCKW[t])/WEEKS

SEASON_POOL=sorted(season_avg(t) for t in TEAMS)

season_pct=lambda t: 100*sum(1 for x in POOL if x < season_avg(t))/len(POOL)

wkrank=lambda d,t,k,hi=True: sorted(TEAMS,key=lambda x:(-d[x][k] if hi else d[x][k])).index(t)+1

LUCK_COLS=[("QB Fault","qb_fault","#9F1B17"),("Drop","drop","#C9302C"),("Short","short","#E5484D"),("Flag","flag","#F2837F"),("Replay","replay","#F8BDBA")]

def pct_color(p):
    # 0 = luckiest (green) to 100 = unluckiest (red)
    i=min(11,int(p/100*12)); return SCALE[i]

def meter(p, mobile=False):
    pos=max(2,min(98,p))
    return f'''<div style="position: relative; height: 14px; border-radius: 7px; background: linear-gradient(90deg, #1F8A4C 0%, #9ACBAD 35%, #EEF0F2 50%, #F0A9A5 65%, #C9302C 100%)">
<span style="position: absolute; left: calc({pos:.0f}% - 9px); top: -5px; width: 18px; height: 24px; border-radius: 5px; background: {NAVY}; box-shadow: 0 0 0 3px #FFFFFF"></span></div>
<div style="display: flex; justify-content: space-between; font-size: 12px; color: {INK3}; margin-top: 6px"><span>Luckiest</span><span>Unluckiest</span></div>'''

def hero(mobile=False):
    pr=me["rank"]; inn=pr<=PLAYOFF_TEAMS
    rk=lambda key: sorted(TEAMS,key=key).index(T0)+1
    pf=sum(SC[T0]); pf_rank=rk(lambda x:-sum(SC[x]))
    pa=me["pa"]; pa_rank=rk(lambda x:-[r for r in rows if r["t"]==x][0]["pa"])
    maxpf=lambda t: sum(SC[t][k]*100/LINEW[t][k] for k in range(WEEKS))
    mpf=maxpf(T0); mpf_rank=rk(lambda x:-maxpf(x))
    ln=100*pf/mpf; ln_rank=rk(lambda x:-sum(SC[x])/maxpf(x))
    sp=season_pct(T0)
    streak=1; last=won(T0,WEEKS-1)
    for k in range(WEEKS-2,-1,-1):
        if won(T0,k)==last: streak+=1
        else: break
    mini="".join(f'<span title="Week {k+1}: {SC[T0][k]:.1f} to {SC[opp[T0][k]][k]:.1f}" style="flex: 1; height: 24px; border-radius: 3px; background: {"#1F8A4C" if won(T0,k) else "#D9534E"}; color: #FFFFFF; font-size: 11px; font-weight: 800; display: flex; align-items: center; justify-content: center">{"W" if won(T0,k) else "L"}</span>' for k in range(WEEKS))
    big=34 if not mobile else 30; small=24 if not mobile else 22
    lab=lambda t: f'<span style="font-size: 14px; font-weight: 700; opacity: 0.85">{t}</span>'
    num=lambda v,sz: f'<span style="font-size: {sz}px; font-weight: 800; line-height: 1">{v}</span>'
    sub=lambda t: f'<span style="font-size: 15px; opacity: 0.9">{t}</span>'
    div='<span style="height: 1px; background: rgba(255,255,255,0.2); margin: 6px 0"></span>'
    tile=lambda inner: f'<div style="display: flex; flex-direction: column; gap: 4px; padding: 18px 20px; border-radius: 12px; background: {NAVY}; color: #FFFFFF">{inner}</div>'
    return (f'<div style="display: grid; grid-template-columns: repeat({2 if mobile else 4}, minmax(0, 1fr)); gap: {10 if mobile else 14}px">'
      + tile(lab("Record")+num(f'{me["w"]}-{me["l"]}',big)+sub(f'{ordn(pr)} place · {"in a playoff spot" if inn else "outside the playoffs"}')+div
             +lab("Season")+f'<div style="display: flex; gap: 3px; margin: 4px 0 2px">{mini}</div>')
      + tile(lab("Points for")+num(f'{pf:.1f}',big)+sub(f'{ordn(pf_rank)} in the league')+div+lab("Points against")+num(f'{pa:.1f}',small)+sub(f'{ordn(pa_rank)} most faced'))
      + tile(lab("Max points for")+num(f'{mpf:.1f}',big)+sub(f'{ordn(mpf_rank)} in the league')+div+lab("Lineup score")+num(f'{ln:.1f}%',small)+sub(f'{ordn(ln_rank)} in the league'))
      + tile(lab("Lost to luck")+num(f'{lost_total:.1f}',big)+sub(f'Worse than {sp:.0f}% of team-weeks')+div+lab("Overturned touchdowns")+num(f'{OVR_TD}',small)+sub(f'Worth {OVR_PTS:.1f} points'))
      + '</div>')

def strip(mobile=False):
    rowsx=[]
    def cell(txt,bg,fg,tip,extra="",sub=""):
        s2=f'<span style="display: block; margin-top: 3px; font-size: 10px; font-weight: 800; letter-spacing: 1px; opacity: 0.85">{sub}</span>' if sub else ""
        return f'<td style="padding: 0"><span title="{tip}" style="display: block; margin: 2px; padding: 8px 2px 6px; min-width: 52px; text-align: center; border-radius: 5px; background: {bg}; color: {fg}; font-weight: 700; line-height: 1{extra}">{txt}{s2}</span></td>'
    def r1(k):
        rk=wk_rank[T0][k]; gold=rk==1; dark=(rk<=3 or rk>=10) and not gold
        return cell(f"{SC[T0][k]:.1f}", GOLD if gold else SCALE[rk-1], NAVY if gold else ("#FFFFFF" if dark else INK), f"{rk} of {N}", f"; box-shadow: {GOLD_EDGE}" if gold else "", "WIN" if won(T0,k) else "LOSS")
    def r2(k):
        rk=wkrank(LINEW,T0,k); dark=rk<=3 or rk>=10
        return cell(f"{LINEW[T0][k]:.0f}%", SCALE[rk-1], "#FFFFFF" if dark else INK, f"{rk} of {N}")
    def r3(k):
        p=pct_worse(LUCKW[T0][k]); dark=p>=75 or p<=25
        return cell(f"{LUCKW[T0][k]:.1f}", pct_color(p), "#FFFFFF" if dark else INK, f"Worse than {p:.0f}% of team-weeks")
    def r4(k):
        rk=wkrank(BENCHW,T0,k,hi=False); dark=rk<=3 or rk>=10
        return cell(f"{BENCHW[T0][k]:.1f}", SCALE[rk-1], "#FFFFFF" if dark else INK, f"{rk} of {N} (fewer is better)")
    labels=[("Score",r1,f"{sum(SC[T0])/WEEKS:.1f}"),("Lineup score",r2,f"{100*sum(SC[T0])/sum(SC[T0][k]*100/LINEW[T0][k] for k in range(WEEKS)):.1f}%"),("Lost to luck",r3,f"{lost_total/WEEKS:.1f}"),("Left on bench",r4,f"{sum(BENCHW[T0])/WEEKS:.1f}")]
    th_=lambda t, extra="": f'<th style="padding: 6px 4px; font-size: 13px; color: {NAVY}; border-bottom: 2px solid {NAVY}; white-space: nowrap{extra}">{t}</th>'
    head=f'<th style="padding: 6px 8px; position: sticky; left: 0; background: #FFFFFF; border-bottom: 2px solid {NAVY}"></th>'+"".join(th_(f"Wk {k+1}") for k in range(WEEKS))+th_("Avg", f"; border-left: 2px solid {NAVY}")
    body="".join(f'<tr><th style="padding: 4px 10px; text-align: left; font-size: 14px; color: {NAVY}; white-space: nowrap; position: sticky; left: 0; background: #FFFFFF">{lab}</th>{"".join(f(k) for k in range(WEEKS))}<td style="padding: 0 10px; text-align: right; font-weight: 800; color: {NAVY}; border-left: 2px solid {NAVY}; white-space: nowrap">{avg}</td></tr>' for lab,f,avg in labels)
    return f'<div style="border: 1px solid {LINE}; border-radius: 10px; overflow-x: auto; padding: 8px"><table style="border-collapse: collapse; font-size: 14px; width: 100%; min-width: 900px"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'

def luck_box(mobile=False):
    sp=season_pct(T0)
    worst_k=max(range(WEEKS),key=lambda k:LUCKW[T0][k]); best_k=min(range(WEEKS),key=lambda k:LUCKW[T0][k])
    segs="".join(f'<span title="{n}" style="flex: {LUCK_SPLIT[c]} 1 0; background: {col}; color: {"#FFFFFF" if i<3 else NAVY}; font-size: 13px; font-weight: 700; display: flex; align-items: center; padding-left: 8px; white-space: nowrap; overflow: hidden; border-right: 2px solid #FFFFFF">{'' if mobile else n}</span>' for i,(n,c,col) in enumerate(LUCK_COLS))
    rows_="".join(f'<div style="display: flex; justify-content: space-between; padding: 8px 0; border-bottom: 1px solid {LINE2}"><span style="display: inline-flex; align-items: center; gap: 8px"><span style="width: 12px; height: 12px; border-radius: 3px; background: {col}"></span>{n}</span><b>{lost_total*LUCK_SPLIT[c]:.1f}</b></div>' for n,c,col in LUCK_COLS)
    callout=lambda lab,k,col: f'<div style="flex: 1; min-width: 150px; padding: 14px 16px; border-radius: 10px; background: {col}; color: #FFFFFF"><span style="font-size: 13px; font-weight: 700; opacity: 0.9">{lab}</span><div style="font-size: 26px; font-weight: 800">Week {k+1}</div><span style="font-size: 15px">{LUCKW[T0][k]:.1f} lost · worse than {pct_worse(LUCKW[T0][k]):.0f}%</span></div>'
    return f'''<section style="background: #FFFFFF; border: 1px solid {LINE}; border-radius: 12px; padding: {"16px" if mobile else "22px"}; display: flex; flex-direction: column; gap: 16px">
<div style="display: flex; align-items: baseline; gap: 12px; flex-wrap: wrap"><span style="font-size: {36 if mobile else 44}px; font-weight: 800; color: {pct_color(sp)}; line-height: 1">{ordn(round(sp))}</span><span style="font-size: 17px; color: {INK2}">percentile for bad luck: an average week here lost more than {sp:.0f}% of all team-weeks this season</span></div>
{meter(sp, mobile)}
<div style="display: flex; height: 30px; border-radius: 6px; overflow: hidden">{segs}</div>
<div style="display: grid; grid-template-columns: repeat({1 if mobile else 2}, minmax(0, 1fr)); column-gap: 36px">{rows_}</div>
<div style="display: flex; gap: 12px; flex-wrap: wrap">{callout("Unluckiest week",worst_k,"#C9302C")}{callout("Luckiest week",best_k,"#1F8A4C")}</div></section>'''

def positions(mobile=False):
    lg=[sum(POS_SPLIT[t][i] for t in TEAMS)/len(TEAMS) for i in range(len(POS_COLORS))]
    out=""
    for i,(p,_) in enumerate(POS_COLORS):
        v=POS_SPLIT[T0][i]; rk=sorted(TEAMS,key=lambda x:-POS_SPLIT[x][i]).index(T0)+1
        diff=v-lg[i]; mx=max(max(POS_SPLIT[t][i] for t in TEAMS), 1)
        gold = rk==1
        badge=f'<span style="font-size: 13px; font-weight: 800; padding: 3px 8px; border-radius: 4px; background: {GOLD if gold else SCALE[rk-1]}; color: {NAVY if gold or 4<=rk<=9 else "#FFFFFF"}{f"; box-shadow: {GOLD_EDGE}" if gold else ""}">{rk} of {N}</span>'
        out+=f'''<div style="display: grid; grid-template-columns: 44px minmax(0, 1fr) {"110px" if mobile else "190px"}; align-items: center; gap: 12px; padding: 8px 0; border-bottom: 1px solid {LINE2}">
<b style="color: {NAVY}">{p}</b>
<span style="position: relative; height: 22px; background: {LINE2}; border-radius: 4px"><span style="position: absolute; left: 0; top: 0; bottom: 0; width: {100*v/mx:.1f}%; border-radius: 4px; background: {"#1F8A4C" if diff>=0 else "#D9534E"}"></span><span title="League average" style="position: absolute; left: {100*lg[i]/mx:.1f}%; top: -4px; bottom: -4px; width: 3px; background: {NAVY}"></span></span>
<span style="display: flex; justify-content: flex-end; align-items: center; gap: 8px; white-space: nowrap"><b>{v:.1f}</b>{"" if mobile else f'<span style="color: {GREEN if diff>=0 else RED}; font-weight: 700">{sg(diff)}</span>'}{badge}</span></div>'''
    return f'''<section style="background: #FFFFFF; border: 1px solid {LINE}; border-radius: 12px; padding: {"14px 16px" if mobile else "18px 22px"}">{out}
<div style="font-size: 13px; color: {INK2}; margin-top: 10px">Navy tick is the league average.</div></section>'''

GOLD_TXT="#D19A00"

def injuries(mobile=False):
    # one row per injury: every week he was out, his points per game against what his replacements scored
    stints=[x if isinstance(x,dict) else dict(start=x[0],end=x[0],name=x[1],pos=x[2],avg=x[3],weeks=[(x[0],x[4],x[5])],dropped=False) for x in INJ]
    lost=sum(sum(x["avg"]-g for _,_,g in x["weeks"]) for x in stints)
    rows_=""
    for x in stints:
        n=len(x["weeks"]); got=sum(g for _,_,g in x["weeks"]); d=got-x["avg"]*n
        wk=f'Wk {x["start"]}' if x["start"]==x["end"] else f'Wk {x["start"]}–{x["end"]}'
        reps=sorted({nm for _,nm,_ in x["weeks"]})
        who=reps[0] if len(reps)==1 else f"{len(reps)} players"
        rep_line=f'<b>{who}</b> <span style="color: {INK2}">{got/n:.1f} a game{f" over {n} games" if n>1 else ""}</span>'
        note=f'<span style="display: block; font-size: 13px; color: {INK3}; margin-top: 2px">Dropped while hurt</span>' if x["dropped"] else ""
        rows_+=f'''<div style="display: grid; grid-template-columns: {"64px minmax(0, 1fr) 60px" if mobile else "84px minmax(0, 1fr) minmax(0, 1fr) 90px"}; align-items: center; gap: 12px; padding: 12px 10px; border-bottom: 1px solid {LINE2}">
<b style="color: {NAVY}">{wk}</b>
<span><span style="font-size: 11px; font-weight: 800; letter-spacing: 1px; padding: 2px 6px; border-radius: 4px; background: {NAVY}; color: #FFFFFF">OUT</span> <b>{x["name"]}</b> <span style="color: {INK2}">{x["pos"]}, {x["avg"]:.1f} a game</span>{note}{"" if not mobile else f'<span style="display: block; color: {INK2}; margin-top: 3px">In his spot: {rep_line}</span>'}</span>
{"" if mobile else f'<span><span style="font-size: 11px; font-weight: 800; letter-spacing: 1px; padding: 2px 6px; border-radius: 4px; background: {WOLF_L}; color: {NAVY}">IN</span> {rep_line}</span>'}
<b style="text-align: right; font-size: 18px; color: {NAVY}">{sg(d)}</b></div>'''
    return f'''<section style="background: #FFFFFF; border: 1px solid {LINE}; border-left: 6px solid {NAVY}; border-radius: 12px; padding: {"14px 12px" if mobile else "18px 22px"}; display: flex; flex-direction: column">
<div style="display: flex; align-items: baseline; gap: 10px; margin: 0 10px 8px"><span style="font-size: 36px; font-weight: 800; color: {NAVY}; line-height: 1">{sg(-lost)}</span><span style="color: {INK2}; font-size: 16px">{"points lost to injuries this season" if lost>0 else "points net from the players filling in for injured starters"}</span></div>{rows_}
<div style="font-size: 13px; color: {INK2}; margin: 10px 10px 0">Each injured starter&#39;s points per start, every week he was out (including after he was dropped while hurt), against the lowest-scoring starter at his position that week, the spot he&#39;d have taken back. Bye weeks are skipped.</div></section>'''

def switcher(mobile=False):
    chips="".join(f'<a href="{team_file(t)}" style="flex: none; padding: 7px 13px; border-radius: 18px; text-decoration: none; font-weight: 700; font-size: 14px; background: {WOLF if t==T0 else "rgba(255,255,255,0.08)"}; color: {NAVY if t==T0 else "#FFFFFF"}; border: 1px solid {"transparent" if t==T0 else "#3C5476"}">{t}</a>' for t in TEAMS)
    return f'<div style="display: flex; gap: 6px; {"overflow-x: auto" if mobile else "flex-wrap: wrap"}; margin-top: 6px">{chips}</div>'

def tband(mobile=False):
    return f'''<section style="position: relative; overflow: hidden; background: {NAVY}; color: #FFFFFF">
<span aria-hidden="true" style="position: absolute; right: {-16 if mobile else -24}px; top: {-20 if mobile else -56}px; font-size: {150 if mobile else 300}px; font-weight: 800; line-height: 1; color: rgba(165,172,175,0.08)">WFR</span>
<div style="position: relative; {"padding: 18px 16px 10px" if mobile else "max-width: 1240px; margin: 0 auto; padding: 24px 24px 14px"}; display: flex; flex-direction: column; gap: 8px">
<span style="font-size: {14 if mobile else 15}px; font-weight: 700; color: {WOLF}; letter-spacing: 1px">Every manager, up close</span>
<h1 style="margin: 0; font-size: {44 if mobile else 64}px; line-height: 0.95; font-weight: 800">[Team {T0}]</h1>
<p style="margin: 0; font-size: {15 if mobile else 18}px; color: #D5D9DB">{MGR[T0]} · {LEAGUE_NAME} · Through Week {WEEK}</p>
{switcher(mobile)}</div>
{waves(50 if mobile else 80, NAVY)}</section>'''

# ---------- candidate sections ----------
card=lambda inner,mobile,pad=None: f'<section style="background: #FFFFFF; border: 1px solid {LINE}; border-radius: 12px; padding: {pad or ("14px 16px" if mobile else "18px 22px")}">{inner}</section>'

def vs_proj(mobile=False):
    d=[SC[T0][k]-PROJ[k] for k in range(WEEKS)]; mx=max(abs(x) for x in d)
    beat=sum(1 for x in d if x>0)
    bars="".join(f'''<div style="display: flex; flex-direction: column; align-items: center; gap: 4px; flex: 1; min-width: 0">
{"" if mobile else f'<span style="font-size: 12px; font-weight: 700; color: {GREEN if x>0 else RED}; height: 16px">{sg(x)}</span>'}
<div style="position: relative; width: 100%; height: 120px"><span style="position: absolute; left: 15%; right: 15%; {"bottom: 50%" if x>0 else "top: 50%"}; height: {50*abs(x)/mx:.0f}%; background: {"#1F8A4C" if x>0 else "#D9534E"}; border-radius: {"4px 4px 0 0" if x>0 else "0 0 4px 4px"}"></span><span style="position: absolute; left: 0; right: 0; top: 50%; height: 2px; background: {NAVY}"></span></div>
<span style="font-size: 12px; color: {INK2}">{k+1}</span></div>''' for k,x in enumerate(d))
    return card(f'''<div style="display: flex; align-items: baseline; gap: 10px; margin-bottom: 8px"><span style="font-size: {30 if mobile else 36}px; font-weight: 800; line-height: 1; color: {NAVY}; white-space: nowrap">{beat} of {WEEKS}</span><span style="color: {INK2}; font-size: 16px">weeks above projection · {sg(sum(d))} total</span></div>
<div style="display: flex; gap: 2px">{bars}</div>''',mobile)

def usage(mobile=False):
    if BEST_ADD is None:
        best=''
    else:
      n,p,tm,how,st,pts,ppg=BEST_ADD
      best=f'''<div style="display: flex; align-items: center; gap: {14 if mobile else 20}px; padding: {"16px" if mobile else "18px 24px"}; border-radius: 12px; background: {NAVY}; color: #FFFFFF; {"grid-column: 1 / -1" }">
<span style="flex: none; border-radius: 50%; box-shadow: 0 0 0 3px {WOLF}">{face(64 if not mobile else 52, name=n)}</span>
<span style="display: flex; flex-direction: column; gap: 3px; min-width: 0; flex: 1"><span style="font-size: 13px; font-weight: 800; letter-spacing: 1px; color: {WOLF}">BEST ADDITION</span><b style="font-size: {22 if mobile else 26}px; line-height: 1.1">{n}</b><span style="font-size: 15px; color: #D5D9DB">{p} · {tm} · {how}</span></span>
<span style="text-align: right; flex: none"><b style="display: block; font-size: {30 if mobile else 38}px; line-height: 1">{pts:.1f}</b><span style="font-size: 14px; color: #D5D9DB">in {st} starts · {ppg:.1f} a start</span></span></div>'''

    stats=USAGE
    t="".join(f'<div style="padding: 16px 18px; border-radius: 12px; border: 1px solid {LINE}; background: #FFFFFF"><span style="font-size: 13px; font-weight: 800; letter-spacing: 1px; color: {INK2}">{l.upper()}</span><div style="font-size: 34px; font-weight: 800; line-height: 1.1; color: {c if c!=WOLF else NAVY}">{v}</div><span style="font-size: 14px; color: {INK2}">{sub}</span></div>' for l,v,sub,c in stats)
    return f'<div style="display: grid; grid-template-columns: repeat({2 if mobile else 4}, minmax(0, 1fr)); gap: 12px">{best}{t}</div>'

def close_games(mobile=False):
    mg=[(k,SC[T0][k]-SC[opp[T0][k]][k]) for k in range(WEEKS)]
    def rec(lim):
        g=[m for _,m in mg if abs(m)<=lim]; w=sum(1 for m in g if m>0); return w,len(g)-w
    lg=lambda lim: sum(sum(1 for k in range(WEEKS) if abs(SC[t][k]-SC[opp[t][k]][k])<=lim) for t in TEAMS)/len(TEAMS)
    tiles=""
    for lim,lab in [(5,"Within 5"),(10,"Within 10"),(20,"Within 20")]:
        w,l=rec(lim); n=w+l
        frac=w/n if n else 0.5
        bar=f'<div style="display: flex; height: 10px; border-radius: 5px; overflow: hidden; margin: 8px 0 6px; background: {LINE2}">' + (f'<span style="flex: {w} 1 0; background: #1F8A4C"></span><span style="flex: {l} 1 0; background: #D9534E"></span>' if n else "") + '</div>'
        tiles+=f'<div style="padding: 16px 18px; border-radius: 12px; border: 1px solid {LINE}; background: #FFFFFF"><span style="font-size: 13px; font-weight: 800; letter-spacing: 1px; color: {INK2}">{lab.upper()}</span><div style="font-size: 34px; font-weight: 800; line-height: 1.1; color: {GREEN if w>l else RED if l>w else NAVY}">{w}-{l}</div>{bar}<span style="font-size: 14px; color: {INK2}">{n} game{"" if n==1 else "s"} · league average {lg(lim):.1f}</span></div>'
    return f'<div style="display: grid; grid-template-columns: repeat({1 if mobile else 3}, minmax(0, 1fr)); gap: 12px">{tiles}</div>'

# ---------- v3 overrides: calmer colors for the three "cost" sections ----------
STRIPE=f"repeating-linear-gradient(135deg, {WOLF} 0 4px, {WOLF_L} 4px 8px)"

def held_down(mobile=False):
    cost=lambda d:(d[4]-POS_AVG[d[1]])*d[3]
    out=""
    for i,d in enumerate(DRAG):
        n,p,tm,st,ppg=d; c=cost(d); avg=POS_AVG[p]; top=i==0
        out+=f'''<div style="display: grid; grid-template-columns: 28px 44px minmax(0, 1fr) {"70px" if mobile else "130px"}; align-items: center; gap: 12px; padding: 10px; border-radius: 8px; background: {WOLF_L if top else "transparent"}; border-bottom: 1px solid {"transparent" if top else LINE2}">
<span style="font-size: 22px; font-weight: 800; color: {NAVY if top else WOLF}">{i+1}</span>{face(44)}
<span style="display: flex; flex-direction: column; gap: 5px; min-width: 0"><span><b style="font-size: 17px">{n}</b> <span style="color: {INK3}">{p} · {tm} · {st} starts</span></span>
<span style="position: relative; display: block; height: 10px; border-radius: 5px; background: {LINE2}"><span style="position: absolute; left: 0; top: 0; bottom: 0; width: {100*ppg/22:.0f}%; border-radius: 5px 0 0 5px; background: {NAVY}"></span><span style="position: absolute; left: {100*ppg/22:.0f}%; top: 0; bottom: 0; width: {100*(avg-ppg)/22:.0f}%; background: {STRIPE}; border-radius: 0 5px 5px 0"></span></span>
<span style="font-size: 13px; color: {INK2}">{ppg:.1f} a start · average {p} starter {avg:.1f}</span></span>
<span style="text-align: right"><b style="font-size: 18px; color: {NAVY}">{sg(c)}</b><span style="display: block; font-size: 13px; color: {INK2}">points</span></span></div>'''
    tot=sum(cost(d) for d in DRAG)
    return f'''<section style="background: #FFFFFF; border: 1px solid {LINE}; border-radius: 12px; padding: {"6px 6px" if mobile else "8px 12px"}; display: flex; flex-direction: column; gap: 4px">
<div style="display: flex; align-items: baseline; gap: 10px; margin: 10px 10px 8px"><span style="font-size: 36px; font-weight: 800; color: {NAVY}; line-height: 1">{sg(tot)}</span><span style="color: {INK2}; font-size: 16px">points behind an average starter at the same spots</span></div>{out}
<div style="font-size: 13px; color: {INK2}; margin: 8px 10px 10px">Striped part is the gap to the league&#39;s average starter at that position. Minimum {MIN_STARTS} starts.</div></section>'''

def carried(mobile=False):
    tot_=sum(SC[T0]); mx=CARRY[0][3]; out=""
    for i,(n,p,tm,pts,g) in enumerate(CARRY):
        out+=f'''<div style="display: grid; grid-template-columns: 28px 44px minmax(0, 1fr) {"70px" if mobile else "130px"}; align-items: center; gap: 12px; padding: 10px; border-radius: 8px; border-bottom: 1px solid {LINE2}">
<span style="font-size: 22px; font-weight: 800; color: {GOLD_TXT if i==0 else WOLF}">{i+1}</span>{face(44, ring=(i==0))}
<span style="display: flex; flex-direction: column; gap: 5px; min-width: 0"><span><b style="font-size: 17px">{n}</b> <span style="color: {INK3}">{p} · {tm} · {g} starts</span></span>
<span style="display: block; height: 10px; border-radius: 5px; background: {LINE2}"><span style="display: block; height: 100%; width: {100*pts/mx:.0f}%; border-radius: 5px; background: {GOLD if i==0 else NAVY}"></span></span>
<span style="font-size: 13px; color: {INK2}">{pts/g:.1f} a start</span></span>
<span style="text-align: right"><b style="font-size: 18px">{pts:.1f}</b><span style="display: block; font-size: 13px; color: {INK2}">{100*pts/tot_:.0f}% of points</span></span></div>'''
    return f'<section style="background: #FFFFFF; border: 1px solid {LINE}; border-radius: 12px; padding: {"6px 6px" if mobile else "8px 12px"}">{out}</section>'

def regrets(mobile=False):
    cards=""
    for i,(wk,b,p,bp,s_,sp_) in enumerate(REGRETS):
        side=lambda tag,n,v,col,tc: f'''<div style="display: flex; flex-direction: column; align-items: center; gap: 6px; flex: 1; min-width: 0; text-align: center">{face(48, name=n)}
<span style="font-size: 11px; font-weight: 800; letter-spacing: 1px; color: {tc}">{tag}</span><b style="font-size: 30px; line-height: 1; color: {col}">{v:.1f}</b><b style="font-size: 15px; line-height: 1.15; min-height: 2.3em">{n}</b></div>'''
        cards+=f'''<div style="display: flex; flex-direction: column; gap: 14px; padding: 16px 18px; border-radius: 12px; border: 1px solid {LINE}; background: #FFFFFF">
<span style="display: flex; justify-content: space-between; align-items: baseline"><b style="color: {NAVY}; font-size: 17px">Week {wk}</b><span style="font-size: 13px; color: {INK2}">{p}</span></span>
<div style="display: flex; align-items: flex-start; gap: 8px">{side("BENCHED",b,bp,NAVY,NAVY)}<span style="margin-top: 92px; font-size: 14px; font-weight: 800; color: {WOLF}">VS.</span>{side("STARTED",s_,sp_,WOLF,INK3)}</div>
<span style="border-top: 1px solid {LINE2}; padding-top: 10px; font-size: 15px; color: {INK2}; text-align: center"><b style="font-size: 22px; color: {NAVY}">{bp-sp_:.1f}</b> left on the bench</span></div>'''
    return f'<div style="display: grid; grid-template-columns: repeat({1 if mobile else 3}, minmax(0, 1fr)); gap: 12px">{cards}</div>'

def highs_lows(mobile=False):
    mg=[SC[T0][k]-SC[opp[T0][k]][k] for k in range(WEEKS)]
    wins=[k for k in range(WEEKS) if mg[k]>0]; losses=[k for k in range(WEEKS) if mg[k]<0]
    bw=max(wins,key=lambda k:mg[k]) if wins else None; cl=max(losses,key=lambda k:mg[k]) if losses else None
    def perf(bg,fg,sub_c,lab,pl,big,line1,line2):
        n,p,tm,wk=pl[:4]
        return f'''<div style="display: flex; align-items: center; gap: 16px; padding: 16px 18px; border-radius: 12px; background: {bg}; color: {fg}"><span style="flex: none; border-radius: 50%; box-shadow: 0 0 0 3px #FFFFFF">{face(56, name=n)}</span>
<span style="display: flex; flex-direction: column; gap: 3px; flex: 1; min-width: 0"><span style="font-size: 13px; font-weight: 800; letter-spacing: 1px; opacity: 0.85">{lab}</span><b style="font-size: 20px; line-height: 1.1">{n} <span style="font-weight: 500; font-size: 15px; color: {sub_c}">{p} · {tm} · Week {wk}</span></b><span style="font-size: 14px; color: {sub_c}">{line2}</span></span>
<span style="text-align: right; flex: none"><b style="display: block; font-size: 34px; line-height: 1">{big}</b><span style="font-size: 13px; color: {sub_c}">{line1}</span></span></div>'''
    small=lambda bg,fg,lab,big,sub: f'<div style="display: flex; flex-direction: column; justify-content: center; padding: 16px 18px; border-radius: 12px; background: {bg}; color: {fg}"><span style="font-size: 13px; font-weight: 800; letter-spacing: 1px; opacity: 0.85">{lab}</span><b style="font-size: 34px; line-height: 1.1">{big}</b><span style="font-size: 14px">{sub}</span></div>'
    b=PERF_BEST; w=PERF_WORST
    tiles=(perf(NAVY,"#FFFFFF","#D5D9DB","BEST PERFORMANCE",b,f"{b[4]:.1f}","points",b[5])
      + (small(NAVY,"#FFFFFF","BIGGEST WIN",sg(mg[bw]),f"Week {bw+1} over [Team {opp[T0][bw]}]") if bw is not None else small(NAVY,"#FFFFFF","BIGGEST WIN","—","No wins yet"))
      + perf(WOLF_L,NAVY,INK2,"WORST PERFORMANCE",w,f"{w[4]:.1f}",f"projected {w[5]:.1f}",w[6])
      + (small(WOLF_L,NAVY,"CLOSEST LOSS",sg(mg[cl]),f"Week {cl+1} to [Team {opp[T0][cl]}]") if cl is not None else small(WOLF_L,NAVY,"CLOSEST LOSS","—","No losses yet")))
    return f'<div style="display: grid; grid-template-columns: {"minmax(0, 1fr)" if mobile else "minmax(0, 3fr) minmax(0, 1fr)"}; gap: 12px">{tiles}</div>'

def teams_page(mobile=False):
    h=lambda t,note="": h2(t,note,mobile)
    body=f'''<main style="{"padding: 4px 16px 32px" if mobile else "max-width: 1240px; margin: 0 auto; padding: 4px 24px 44px"}; display: flex; flex-direction: column; gap: {22 if mobile else 28}px">
{hero(mobile)}
<section style="display: flex; flex-direction: column; gap: 12px">{h("Week by week")}{strip(mobile)}</section>
<section style="display: flex; flex-direction: column; gap: 12px">{h("Luck")}{luck_box(mobile)}</section>
<section style="display: flex; flex-direction: column; gap: 12px">{h("Points by position")}{positions(mobile)}</section>
<section style="display: flex; flex-direction: column; gap: 12px">{h("Who carried the team")}{carried(mobile)}</section>
<section style="display: flex; flex-direction: column; gap: 12px">{h("Who held the team down")}{held_down(mobile)}</section>
<section style="display: flex; flex-direction: column; gap: 12px">{h("Bench regrets")}{regrets(mobile)}</section>
<section style="display: flex; flex-direction: column; gap: 12px">{h("Injuries")}{injuries(mobile)}</section>
<section style="display: flex; flex-direction: column; gap: 12px">{h("Highs and lows")}{highs_lows(mobile)}</section>
<section style="display: flex; flex-direction: column; gap: 12px">{h("Close games")}{close_games(mobile)}</section>
<section style="display: flex; flex-direction: column; gap: 12px">{h("Vs. projection")}{vs_proj(mobile)}</section>
<section style="display: flex; flex-direction: column; gap: 12px">{h("Roster moves")}{usage(mobile)}</section>
</main>'''
    root = 'width: 390px; min-height: 6900px' if mobile else 'min-height: 100%'
    return f'''<div style="{root}; background: #FFFFFF; color: {INK}; font-family: {F}; font-size: 15px">
{navbar(active="Teams", league=True, mobile=mobile)}{tband(mobile)}{body}{footer(mobile)}</div>'''

for t in TEAMS:
    globals().update(D["teams"][t])
    write_page(team_file(t), f"[Team {t}]", teams_page(), teams_page(True), active="Teams")
# teams.html opens the first team's page, so pages that don't know the team IDs (like Almost Points) can link to Teams
(SITE / "teams.html").write_text(f'<!doctype html><meta charset="utf-8"><meta http-equiv="refresh" content="0; url={team_file(TEAMS[0])}"><a href="{team_file(TEAMS[0])}">Teams</a>', encoding="utf-8")
print(len(TEAMS), "team pages written")

## Draft & Moves

The draft board, report card, steals and busts, trades, and the waiver wire.

In [ ]:
INJ_ICON='<svg width="14" height="14" viewBox="0 0 14 14" aria-label="Injured"><circle cx="7" cy="7" r="6.25" fill="#C9302C" stroke="#FFFFFF" stroke-width="1.5"/><path d="M7 3.5v7M3.5 7h7" stroke="#FFFFFF" stroke-width="2" stroke-linecap="round"/></svg>'

CUTS=[(7,"A+"),(5,"A"),(3.5,"A-"),(2.5,"B+"),(1.5,"B"),(0.75,"B-"),(0,"C+"),(-0.75,"C"),(-1.5,"C-"),(-2.25,"D+"),(-3,"D"),(-4,"D-")]   # points a week

def LETP(g):
    for c,l in CUTS:
        if g>=c: return l
    return "F"

LET=lambda q: LETP(q["gap"])

GRADES=["A+","A","A-","B+","B","B-","C+","C","C-","D+","D","D-","F"]

GC=["#1F8A4C","#2E9558","#4FA971","#7DBF96","#A8D3B8","#CFE5D7","#EEF0F2","#EEF0F2","#EEF0F2","#F6D2D0","#F0A9A5","#E06D68","#C9302C"]

GCOL={g:(GC[i],"#FFFFFF" if i<=2 or i>=11 else INK) for i,g in enumerate(GRADES)}

GPTS={g:(12-i)/3 for i,g in enumerate(GRADES)}     # A+ 4.0 ... F 0

def gchip(l,big=False,gold=False):
    bg,fg=GCOL[l]
    if gold: bg,fg=GOLD,NAVY
    return f'<span style="display: inline-flex; align-items: center; justify-content: center; min-width: {44 if big else 26}px; height: {44 if big else 22}px; padding: 0 {6 if big else 4}px; border-radius: {8 if big else 5}px; background: {bg}; color: {fg}; font-weight: 800; font-size: {22 if big else 13}px{f"; box-shadow: {GOLD_EDGE}" if gold else ""}">{l}</span>'

def team_gpa(t):
    g=[GPTS[LET(q)] for q in picks if q["t"]==t]; return sum(g)/len(g)

def gpa_letter(x):
    return GRADES[min(12,max(0,round(12-3*x)))]

def gpa_col(x): return GCOL[gpa_letter(x)]

rp=lambda q: f"{q['r']}.{q['p']:02d}"

pos_col={"QB":"#13294B","RB":"#3C5476","WR":"#6E7F96","TE":"#A5ACAF","K":"#C9CED1","DEF":"#C9CED1"}

def short(n):
    parts=n.split(" ")
    if n.endswith("DEF"): return n.replace(" DEF","")
    return parts[0][0]+". "+" ".join(parts[1:]) if len(parts)>1 else n

def d_hero(mobile=False):
    gp=sorted(TEAMS,key=lambda t:-team_gpa(t)); best=gp[0]; worst=gp[-1]
    pl=[q for q in picks if q["pos"] not in ("K","DEF")]; steal=max(pl,key=lambda q:q["gap"]); bust=min([q for q in pl if q["r"]<=7 and not q["inj"]],key=lambda q:q["gap"])
    lab=lambda t: f'<span style="font-size: 14px; font-weight: 700; opacity: 0.85">{t}</span>'
    num=lambda v,sz=34: f'<span style="font-size: {sz if not mobile else sz-4}px; font-weight: 800; line-height: 1">{v}</span>'
    sub=lambda t: f'<span style="font-size: 15px; opacity: 0.9">{t}</span>'
    tile=lambda inner: f'<div style="display: flex; flex-direction: column; gap: 4px; padding: 18px 20px; border-radius: 12px; background: {NAVY}; color: #FFFFFF">{inner}</div>'
    return (f'<div style="display: grid; grid-template-columns: repeat({2 if mobile else 4}, minmax(0, 1fr)); gap: {10 if mobile else 14}px">'
      + tile(lab("Best draft")+num(f"[Team {best}]",28)+sub(f"{sum(q['pts'] for q in picks if q['t']==best):.0f} points from picks"))
      + tile(lab("Worst draft")+num(f"[Team {worst}]",28)+sub(f"{sum(q['pts'] for q in picks if q['t']==worst):.0f} points from picks"))
      + tile(lab("Steal of the draft")+num(steal["name"],24)+sub(f"Pick {rp(steal)} · drafted as {steal['drank']}, performing as {steal['prank']}"))
      + tile(lab("Biggest bust")+num(bust["name"],24)+sub(f"Pick {rp(bust)} · drafted as {bust['drank']}, performing as {bust['prank']}"))
      + '</div>')

def board(mobile=False):
    head="".join(f'<th style="padding: 8px 4px; font-size: 14px; color: {NAVY}; border-bottom: 2px solid {NAVY}; white-space: nowrap">{t}</th>' for t in ORDER)
    body=""
    for r in range(1,ROUNDS+1):
        cells=""
        for t in ORDER:
            q=[q for q in picks if q["r"]==r and q["t"]==t][0]; l=LET(q); bg,fg=GCOL[l]
            cells+=f'''<td style="padding: 2px"><div title="{q['name']} · {q['pts']:.1f} points" style="display: flex; flex-direction: column; gap: 2px; padding: 6px 7px; min-width: {84 if mobile else 0}px; border-radius: 6px; background: {bg}; color: {fg}">
<span style="display: flex; justify-content: space-between; font-size: 11px; font-weight: 800; opacity: 0.85"><span>{q['pos']} · {NFL[q['name']]}</span><span>{l}</span></span><span style="font-size: 13px; font-weight: 700; white-space: nowrap; overflow: hidden; text-overflow: ellipsis">{short(q['name'])}</span><span style="display: flex; justify-content: space-between; align-items: center; font-size: 12px"><span style="opacity: 0.85">{q['pts']:.1f}</span>{INJ_ICON if q["inj"] else ""}</span></div></td>'''
        body+=f'<tr><th style="padding: 4px 10px 4px 4px; font-size: 13px; color: {NAVY}; position: sticky; left: 0; background: #FFFFFF">R{r}</th>{cells}</tr>'
    legend=f'<div style="display: flex; align-items: center; gap: 6px; font-size: 13px; color: {INK2}; margin-top: 8px">{INJ_ICON} Out 5 or more weeks, or on IR and still out. Never counted as a bust or a worst pick.</div>'
    return f'''<div style="border: 1px solid {LINE}; border-radius: 12px; padding: 10px; overflow-x: auto"><table style="border-collapse: collapse; width: 100%; table-layout: {"auto" if mobile else "fixed"}"><thead><tr><th style="width: 36px; position: sticky; left: 0; background: #FFFFFF; border-bottom: 2px solid {NAVY}"></th>{head}</tr></thead><tbody>{body}</tbody></table></div>
{legend}'''

def report(mobile=False):
    order=[r["t"] for r in rows]
    best_gpa=max(team_gpa(t) for t in TEAMS)
    out=""
    for t in order:
        tp=[q for q in picks if q["t"]==t]; g=team_gpa(t); bg,fg=gpa_col(g); gold=g==best_gpa
        hi=max(tp,key=lambda q:q["gap"]); lo=min([q for q in tp if not q["inj"]] or tp,key=lambda q:q["gap"])
        dist="".join(f'<span style="flex: 1; height: 10px; border-radius: 2px; background: {GCOL[LET(q)][0]}" title="R{q["r"]} {q["name"]}"></span>' for q in tp)
        chip=f'<span style="display: inline-flex; align-items: center; justify-content: center; width: 52px; height: 40px; border-radius: 8px; background: {GOLD if gold else bg}; color: {NAVY if gold else fg}; font-weight: 800; font-size: 20px{f"; box-shadow: {GOLD_EDGE}" if gold else ""}">{gpa_letter(g)}</span>'
        if mobile:
            out+=f'''<div style="display: grid; grid-template-columns: minmax(0, 1fr); gap: 12px; align-items: center; padding: 10px 0; border-bottom: 1px solid {LINE2}">
<span style="display: flex; flex-direction: column; gap: 5px"><b style="color: {NAVY}">[Team {t}]</b><span style="display: flex; gap: 2px">{dist}</span><span style="font-size: 13px; color: {INK2}">Best: {hi['name']}, {rp(hi)}</span></span></div>'''
        else:
            out+=f'''<div style="display: grid; grid-template-columns: 130px minmax(0, 1.2fr) 90px minmax(0, 1fr) minmax(0, 1fr); gap: 16px; align-items: center; padding: 10px 0; border-bottom: 1px solid {LINE2}">
<b style="color: {NAVY}">[Team {t}]</b><span style="display: flex; gap: 2px">{dist}</span><b style="text-align: right">{sum(q['pts'] for q in tp):.1f}</b>
<span style="font-size: 14px"><span style="font-size: 11px; font-weight: 800; letter-spacing: 1px; color: {GREEN}">BEST</span> {hi['name']} <span style="color: {INK2}; font-weight: 700">{rp(hi)}</span></span>
<span style="font-size: 14px"><span style="font-size: 11px; font-weight: 800; letter-spacing: 1px; color: {RED}">WORST</span> {lo['name']} <span style="color: {INK2}; font-weight: 700">{rp(lo)}</span></span></div>'''
    head="" if mobile else f'<div style="display: grid; grid-template-columns: 130px minmax(0, 1.2fr) 90px minmax(0, 1fr) minmax(0, 1fr); gap: 16px; padding-bottom: 8px; border-bottom: 2px solid {NAVY}; font-size: 13px; font-weight: 700; color: {NAVY}"><span>Team</span><span>Every pick, R1 to R{ROUNDS}</span><span style="text-align: right">Points</span><span>Best pick</span><span>Worst pick</span></div>'
    return card(head+out,mobile)

def steals_busts(mobile=False):
    pl=[q for q in picks if q["pos"] not in ("K","DEF")]
    st=sorted(pl,key=lambda q:-q["gap"])[:5]; bu=sorted([q for q in pl if q["r"]<=7 and not q["inj"]],key=lambda q:q["gap"])[:5]
    def col(title,lst,c):
        r_="".join(f'''<div style="display: grid; grid-template-columns: 40px minmax(0, 1fr) {"64px" if mobile else "92px"}; gap: 10px; align-items: center; padding: 10px 0; border-bottom: 1px solid {LINE2}">{face(40)}
<span style="display: flex; flex-direction: column; gap: 3px; min-width: 0"><span><b>{q['name']}</b> <span style="color: {INK3}">{NFL[q['name']]} · {rp(q)} · [Team {q['t']}]</span></span>
<span style="font-size: 14px; color: {INK2}">Drafted as <b style="color: {NAVY}">{q['drank']}</b> · performing as <b style="color: {c}">{q['prank']}</b></span></span>
<span style="display: flex; align-items: center; justify-content: flex-end; gap: 8px">{"" if mobile else f'<b style="color: {c}; white-space: nowrap">{sg(q["gap"])}/wk</b>'}{gchip(LET(q))}</span></div>''' for q in lst)
        return f'<div style="display: flex; flex-direction: column"><span style="font-size: 13px; font-weight: 800; letter-spacing: 1px; color: {c}; margin-bottom: 4px">{title}</span>{r_}</div>'
    return card(f'''<div style="display: grid; grid-template-columns: repeat({1 if mobile else 2}, minmax(0, 1fr)); gap: {20 if mobile else 36}px">{col("STEALS",st,GREEN)}{col("BUSTS",bu,RED)}</div>
''',mobile)

# trade grade: lineup points with the trade minus lineup points without it, per week since
TCUTS=[(10,"A+"),(7.5,"A"),(5.5,"A-"),(4,"B+"),(2.5,"B"),(1.5,"B-"),(0.5,"C+"),(-1.5,"C"),(-2.5,"C-"),(-4,"D+"),(-5.5,"D"),(-7.5,"D-")]

def TLET(x):
    for c,l in TCUTS:
        if x>=c: return l
    return "F"

def trades(mobile=False):
    out=""
    if not TRADES:
        return f'<div style="font-size: 15px; color: {INK2}">No trades yet this season.</div>'
    for T in TRADES:
        weeks=WEEKS-T["wk"]+1
        def side(t,rec,w,wo):
            d=w-wo; l=TLET(d/weeks)
            pl="".join(f'<span style="display: inline-flex; gap: 6px; align-items: baseline"><b>{n}</b><span style="color: {INK3}">{p} · {NFL[n]}</span></span>' for n,p in rec)
            ln=lambda lab,v,b=False: f'<div style="display: flex; justify-content: space-between; padding: 6px 0; border-bottom: 1px solid {LINE2}"><span style="color: {INK2}">{lab}</span><b>{v:.1f}</b></div>'
            return f'''<div style="flex: 1; min-width: 0; display: flex; flex-direction: column; gap: 8px"><span style="display: flex; justify-content: space-between; align-items: center"><b style="color: {NAVY}; font-size: 17px">[Team {t}]</b>{gchip(l,True)}</span>
<span style="display: flex; flex-direction: column; gap: 2px"><span style="font-size: 11px; font-weight: 800; letter-spacing: 1px; color: {INK3}">GOT</span>{pl}</span>
<div>{ln("Lineup points with the trade",w)}{ln("Lineup points without it",wo)}</div>
<span style="display: flex; justify-content: space-between; align-items: baseline"><span style="color: {INK2}">{"Gained" if d>=0 else "Lost"}</span><b style="font-size: 22px; color: {GREEN if d>=0 else RED}">{sg(d)}</b></span></div>'''
        out+=f'''<div style="padding: 16px 18px; border: 1px solid {LINE}; border-radius: 12px; display: flex; flex-direction: column; gap: 12px">
<span style="font-size: 13px; font-weight: 800; letter-spacing: 1px; color: {INK2}">WEEK {T["wk"]} TRADE · {weeks} WEEKS SINCE</span>
<div style="display: flex; {"flex-direction: column; gap: 18px" if mobile else "gap: 28px"}">{side(T["a"],T["ra"],T["a_with"],T["a_without"])}<span style="{"display: none" if mobile else f"width: 1px; background: {LINE}"}"></span>{side(T["b"],T["rb"],T["b_with"],T["b_without"])}</div></div>'''
    return f'''<div style="display: grid; grid-template-columns: repeat({1 if mobile else 2}, minmax(0, 1fr)); gap: 12px">{out}</div>
<div style="font-size: 13px; color: {INK2}">Best possible lineup points since the trade, with and without it.</div>'''

def waivers(mobile=False):
    pp=[a[4]/a[5] for a in ADDS]; mx=max(pp) if pp else 1
    def wl(i):
        pct=100*(len(ADDS)-1-i)/max(len(ADDS)-1,1); return GRADES[min(12,int((100-pct)/100*13))]
    rs="".join(f'''<div style="display: grid; grid-template-columns: {"40px minmax(0, 1fr) 64px 34px" if mobile else "28px 40px minmax(0, 1fr) 150px 34px"}; gap: 10px; align-items: center; padding: 9px 0; border-bottom: 1px solid {LINE2}">
{"" if mobile else f'<span style="font-size: 20px; font-weight: 800; color: {WOLF}">{i+1}</span>'}{face(40)}
<span style="display: flex; flex-direction: column; gap: 4px; min-width: 0"><span><b>{n}</b> <span style="color: {INK3}">{p} · {NFL[n]} · [Team {t}] · Week {wk}</span></span><span style="display: block; height: 8px; border-radius: 4px; background: {LINE2}"><span style="display: block; height: 100%; width: {100*(pts/st)/mx:.0f}%; border-radius: 4px; background: {NAVY}"></span></span></span>
<span style="text-align: right"><b>{pts/st:.1f}</b><span style="display: block; font-size: 12px; color: {INK2}">a start · {st} starts</span></span>{gchip(wl(i))}</div>''' for i,(n,p,t,wk,pts,st) in enumerate(ADDS))
    order=sorted(TEAMS,key=lambda t:-MOVES[t]); mm=MOVES[order[0]]
    bars="".join(f'<div style="display: grid; grid-template-columns: 70px minmax(0, 1fr) 30px; gap: 10px; align-items: center; padding: 4px 0"><b style="color: {NAVY}; font-size: 14px">Team {t}</b><span style="display: block; height: 14px; border-radius: 4px; background: {LINE2}"><span style="display: block; height: 100%; width: {100*MOVES[t]/mm:.0f}%; border-radius: 4px; background: {NAVY if t==T0 else WOLF}"></span></span><b style="text-align: right; font-size: 14px">{MOVES[t]}</b></div>' for t in order)
    return f'''<div style="display: grid; grid-template-columns: {"minmax(0, 1fr)" if mobile else "minmax(0, 2fr) minmax(0, 1fr)"}; gap: 12px">
{card(f'<span style="font-size: 13px; font-weight: 800; letter-spacing: 1px; color: {INK2}">BEST PICKUPS</span>{rs}<div style="font-size: 13px; color: {INK2}; margin-top: 8px">Points per start since the add. Graded against every pickup this season.</div>',mobile)}
{card(f'<span style="font-size: 13px; font-weight: 800; letter-spacing: 1px; color: {INK2}; display: block; margin-bottom: 6px">MOVES BY TEAM</span>{bars}',mobile)}</div>'''

def dband(mobile=False):
    return f'''<section style="position: relative; overflow: hidden; background: {NAVY}; color: #FFFFFF">
<span aria-hidden="true" style="position: absolute; right: {-16 if mobile else -24}px; top: {-20 if mobile else -56}px; font-size: {150 if mobile else 300}px; font-weight: 800; line-height: 1; color: rgba(165,172,175,0.08)">WFR</span>
<div style="position: relative; {"padding: 18px 16px 10px" if mobile else "max-width: 1240px; margin: 0 auto; padding: 24px 24px 14px"}; display: flex; flex-direction: column; gap: 8px">
<span style="font-size: {14 if mobile else 15}px; font-weight: 700; color: {WOLF}; letter-spacing: 1px">Every pick, graded in hindsight</span>
<h1 style="margin: 0; font-size: {44 if mobile else 64}px; line-height: 0.95; font-weight: 800">Draft &amp; Moves</h1>
<p style="margin: 0; font-size: {15 if mobile else 18}px; color: #D5D9DB">{LEAGUE_NAME} · Through Week {WEEK}</p></div>
{waves(50 if mobile else 80, NAVY)}</section>'''

def draft_page(mobile=False):
    h=lambda t,note="": h2(t,note,mobile)
    sec=lambda t,inner: f'<section style="display: flex; flex-direction: column; gap: 12px">{h(t)}{inner}</section>'
    body=f'''<main style="{"padding: 4px 16px 32px" if mobile else "max-width: 1240px; margin: 0 auto; padding: 4px 24px 44px"}; display: flex; flex-direction: column; gap: {22 if mobile else 28}px">
{d_hero(mobile)}
{sec("Draft board",board(mobile))}
{sec("Draft report card",report(mobile))}
{sec("Steals and busts",steals_busts(mobile))}
{sec("Trades",trades(mobile))}
{sec("Waiver wire",waivers(mobile))}
</main>'''
    root='width: 390px; min-height: 6300px' if mobile else 'min-height: 100%'
    return f'''<div style="{root}; background: #FFFFFF; color: {INK}; font-family: {F}; font-size: 15px">
{navbar(active="Draft &amp; Moves", league=True, mobile=mobile)}{dband(mobile)}{body}{footer(mobile)}</div>'''

write_page("draft.html", "Draft & Moves", draft_page(), draft_page(True), active="Draft &amp; Moves")
print("Draft & Moves written")

## History

Champions, the all-time standings, head to head, the record book, rivalries, every finish, playoff records, and season superlatives.

In [ ]:
def alltime(t):
    W=sum(HIST[y]["w"][t] for y in SEASONS)+CUR[t][0]; L=sum(HIST[y]["G"]-HIST[y]["w"][t] for y in SEASONS)+CUR[t][1]
    PF=sum(sum(HIST[y]["sc"][t]) for y in SEASONS)+CUR[t][2]
    return dict(t=t,W=W,L=L,PF=PF,titles=sum(1 for y in SEASONS if HIST[y]["champ"]==t),
                fin=sum(1 for y in SEASONS if HIST[y]["runner"]==t),po=sum(1 for y in SEASONS if t in HIST[y]["po"]),
                last=sum(1 for y in SEASONS if HIST[y]["last"]==t))

AT=sorted([alltime(t) for t in TEAMS],key=lambda a:(-a["titles"],-a["W"]/(a["W"]+a["L"])))

TROPHY='<svg width="{s}" height="{s}" viewBox="0 0 24 24" aria-hidden="true"><path d="M7 3h10v3h3a3 3 0 0 1-3 4.6A5 5 0 0 1 13 14v3h3v3H8v-3h3v-3a5 5 0 0 1-4-3.4A3 3 0 0 1 4 6h3z" fill="{c}"/></svg>'

troph=lambda s=20,c="#FFC300": TROPHY.format(s=s,c=c)

def h_hero(mobile=False):
    most=max(AT,key=lambda a:a["titles"]); best=max(AT,key=lambda a:a["W"]/(a["W"]+a["L"]))
    allw=[(y,k,t,HIST[y]["sc"][t][k]) for y in SEASONS for t in TEAMS for k in range(HIST[y]["G"])]
    hi=max(allw,key=lambda x:x[3])
    lab=lambda t: f'<span style="font-size: 14px; font-weight: 700; opacity: 0.85">{t}</span>'
    num=lambda v,sz=30: f'<span style="font-size: {sz if not mobile else sz-4}px; font-weight: 800; line-height: 1">{v}</span>'
    sub=lambda t: f'<span style="font-size: 15px; opacity: 0.9">{t}</span>'
    tile=lambda inner: f'<div style="display: flex; flex-direction: column; gap: 4px; padding: 18px 20px; border-radius: 12px; background: {NAVY}; color: #FFFFFF">{inner}</div>'
    return (f'<div style="display: grid; grid-template-columns: repeat({2 if mobile else 4}, minmax(0, 1fr)); gap: {10 if mobile else 14}px">'
      + tile(lab("Seasons")+num(f"{len(SEASONS)+1}",34)+sub(f"{SEASONS[0]} to 2026"))
      + tile(lab("Most titles")+num(f"[Team {most['t']}]",26)+sub(f"{most['titles']} championships"))
      + tile(lab("Best win rate")+num(f"[Team {best['t']}]",26)+sub(f"{best['W']}-{best['L']} all time · {100*best['W']/(best['W']+best['L']):.0f}%"))
      + tile(lab("Highest score")+num(f"{hi[3]:.1f}",34)+sub(f"[Team {hi[2]}] · Week {hi[1]+1}, {hi[0]}"))
      + '</div>')

def champions(mobile=False):
    cards=""
    for y in SEASONS[::-1]:
        H=HIST[y]
        cards+=f'''<div style="display: flex; flex-direction: column; gap: 10px; padding: 18px; border-radius: 12px; background: {GOLD}; box-shadow: {GOLD_EDGE}; color: {NAVY}">
<span style="display: flex; justify-content: space-between; align-items: center"><b style="font-size: 20px">{y}</b>{troph(26,NAVY)}</span>
<span style="display: flex; flex-direction: column"><span style="font-size: 12px; font-weight: 800; letter-spacing: 1px; opacity: 0.75">CHAMPION</span><b style="font-size: 26px; line-height: 1.1">[Team {H["champ"]}]</b><span style="font-size: 14px">{H["w"][H["champ"]]}-{H["G"]-H["w"][H["champ"]]} · {H["stand"].index(H["champ"])+1} seed</span></span>
<span style="height: 1px; background: rgba(19,41,75,0.25)"></span>
<span style="font-size: 14px; display: flex; flex-direction: column; gap: 2px"><span><b>Runner-up</b> [Team {H["runner"]}]</span><span><b>Best record</b> [Team {H["stand"][0]}] ({H["w"][H["stand"][0]]}-{H["G"]-H["w"][H["stand"][0]]})</span><span><b>Last place</b> [Team {H["last"]}]</span></span></div>'''
    cur_card=f'''<div style="display: flex; flex-direction: column; gap: 10px; padding: 18px; border-radius: 12px; border: 2px dashed {WOLF}; color: {NAVY}; justify-content: center; align-items: center; text-align: center">
<b style="font-size: 20px">2026</b><span style="font-size: 15px; color: {INK2}">In progress. [Team {rows[0]["t"]}] leads at {rows[0]["w"]}-{rows[0]["l"]}.</span></div>'''
    return f'<div style="display: grid; grid-template-columns: repeat({1 if mobile else 5}, minmax(0, 1fr)); gap: 12px">{cur_card}{cards}</div>'

def alltime_table(mobile=False):
    th_=lambda t,al="center": f'<th style="padding: 8px 6px; font-size: 13px; color: {NAVY}; border-bottom: 2px solid {NAVY}; text-align: {al}; white-space: nowrap">{t}</th>'
    pct=lambda a: a["W"]/(a["W"]+a["L"])
    ranks=sorted(AT,key=lambda a:-pct(a)); rk={a["t"]:i+1 for i,a in enumerate(ranks)}
    pfr=sorted(AT,key=lambda a:-a["PF"]); pk={a["t"]:i+1 for i,a in enumerate(pfr)}
    body=""
    for i,a in enumerate(AT):
        r=rk[a["t"]]; dark=r<=3 or r>=10
        tr="".join(troph(18) for _ in range(a["titles"]))
        cell=lambda v,bg,fg: f'<td style="padding: 3px"><span style="display: block; padding: 7px 4px; border-radius: 5px; text-align: center; font-weight: 700; background: {bg}; color: {fg}">{v}</span></td>'
        body+=f'''<tr style="border-bottom: 1px solid {LINE2}"><td style="padding: 8px 6px; color: {INK3}; font-weight: 700">{i+1}</td><td style="padding: 8px 6px"><b style="color: {NAVY}">[Team {a["t"]}]</b></td>
<td style="padding: 8px 6px; white-space: nowrap">{tr or f'<span style="color: {LINE}">·</span>'}</td>
{cell(f'{a["W"]}-{a["L"]}',SCALE[r-1],"#FFFFFF" if dark else INK)}{cell(f'{100*pct(a):.0f}%',SCALE[r-1],"#FFFFFF" if dark else INK)}
<td style="padding: 8px 6px; text-align: right">{a["PF"]:,.0f}</td><td style="padding: 8px 6px; text-align: center">{a["po"]}</td><td style="padding: 8px 6px; text-align: center">{a["fin"]+a["titles"]}</td><td style="padding: 8px 6px; text-align: center; color: {RED if a["last"] else INK3}">{a["last"]}</td></tr>'''
    return f'''<div style="border: 1px solid {LINE}; border-radius: 12px; padding: 8px 12px; overflow-x: auto"><table style="border-collapse: collapse; width: 100%; min-width: 760px; font-size: 15px"><thead><tr>{th_("","left")}{th_("Manager","left")}{th_("Titles","left")}{th_("Record")}{th_("Win %")}{th_("Points for","right")}{th_("Playoffs")}{th_("Finals")}{th_("Last place")}</tr></thead><tbody>{body}</tbody></table></div>'''

def h2h(mobile=False):
    rec={}
    for y in SEASONS+[SEASON]:
        for (a,b),gs in H2ALL[y].items():
            for _,_,sa,sb in gs:
                rec.setdefault((a,b),[0,0]); rec.setdefault((b,a),[0,0])
                if sa>sb: rec[(a,b)][0]+=1; rec[(b,a)][1]+=1
                else: rec[(a,b)][1]+=1; rec[(b,a)][0]+=1
    order=[a["t"] for a in AT]
    head="".join(f'<th style="padding: 6px 2px; font-size: 13px; color: {NAVY}; border-bottom: 2px solid {NAVY}">{t}</th>' for t in order)
    body=""
    for a in order:
        cells=""
        for b in order:
            if a==b: cells+=f'<td style="padding: 2px"><span style="display: block; height: 34px; border-radius: 5px; background: {NAVY}"></span></td>'; continue
            w,l=rec.get((a,b),[0,0]); n=w+l
            if n==0: cells+=f'<td style="padding: 2px"><span style="display: flex; align-items: center; justify-content: center; height: 34px; border-radius: 5px; background: {LINE2}; color: {INK3}; font-size: 12px">–</span></td>'; continue
            p=w/n; i=max(0,min(11,int((1-p)*12))); dark=i<=2 or i>=9
            cells+=f'<td style="padding: 2px"><span title="[Team {a}] vs [Team {b}]" style="display: flex; align-items: center; justify-content: center; height: 34px; border-radius: 5px; background: {SCALE[i]}; color: {"#FFFFFF" if dark else INK}; font-size: 13px; font-weight: 700">{w}-{l}</span></td>'
        body+=f'<tr><th style="padding: 2px 10px 2px 0; text-align: left; font-size: 14px; color: {NAVY}; white-space: nowrap; position: sticky; left: 0; background: #FFFFFF">Team {a}</th>{cells}</tr>'
    return f'''<div style="border: 1px solid {LINE}; border-radius: 12px; padding: 10px; overflow-x: auto"><table style="border-collapse: collapse; width: 100%; min-width: 760px; table-layout: fixed"><thead><tr><th style="width: 74px"></th>{head}</tr></thead><tbody>{body}</tbody></table></div>
<div style="font-size: 13px; color: {INK2}">Read across: the row manager&#39;s record against each column manager, every season.</div>'''

def records(mobile=False):
    allw=[(y,k,t,HIST[y]["sc"][t][k]) for y in SEASONS for t in TEAMS for k in range(HIST[y]["G"])]
    games=[(y,k,a,b,sa,sb) for y in SEASONS for (a,b),gs in HIST[y]["h2"].items() for (_,k,sa,sb) in gs]
    blow=max(games,key=lambda g:abs(g[4]-g[5])); close=min([g for g in games if abs(g[4]-g[5])>=0.01],key=lambda g:abs(g[4]-g[5]))
    lo=min(allw,key=lambda x:x[3])
    seasons=[(y,t,sum(HIST[y]["sc"][t]),HIST[y]["w"][t]) for y in SEASONS for t in TEAMS]
    bestpf=max(seasons,key=lambda s:s[2]); bestrec=max(seasons,key=lambda s:(s[3]/HIST[s[0]]["G"],s[2])); worst=min(seasons,key=lambda s:(s[3]/HIST[s[0]]["G"],s[2]))
    win=lambda g:(g[2],g[3]) if g[4]>g[5] else (g[3],g[2])
    items=[("Highest score",f"{max(allw,key=lambda x:x[3])[3]:.1f}",lambda: (lambda h:f"[Team {h[2]}] · Week {h[1]+1}, {h[0]}")(max(allw,key=lambda x:x[3])),True),
           ("Lowest score",f"{lo[3]:.1f}",lambda: f"[Team {lo[2]}] · Week {lo[1]+1}, {lo[0]}",False),
           ("Biggest blowout",f"{abs(blow[4]-blow[5]):.1f}",lambda: f"[Team {win(blow)[0]}] over [Team {win(blow)[1]}] · {blow[0]}",True),
           ("Closest game",f"{abs(close[4]-close[5]):.2f}",lambda: f"[Team {win(close)[0]}] over [Team {win(close)[1]}] · {close[0]}",False),
           ("Most points, one season",f"{bestpf[2]:,.1f}",lambda: f"[Team {bestpf[1]}] · {bestpf[0]}",True),
           ("Best record",f"{bestrec[3]}-{HIST[bestrec[0]]['G']-bestrec[3]}",lambda: f"[Team {bestrec[1]}] · {bestrec[0]}",True),
           ("Worst record",f"{worst[3]}-{HIST[worst[0]]['G']-worst[3]}",lambda: f"[Team {worst[1]}] · {worst[0]}",False),
           ("Longest win streak",str(STREAK[0]),lambda: f"[Team {STREAK[1]}] · {STREAK[2]}"+(f" to {STREAK[3]}" if STREAK[3]!=STREAK[2] else ""),True)]
    t=lambda lab,big,sub,good: f'<div style="padding: 16px 18px; border-radius: 12px; background: {NAVY if good else WOLF_L}; color: {"#FFFFFF" if good else NAVY}"><span style="font-size: 13px; font-weight: 800; letter-spacing: 1px; opacity: 0.85">{lab.upper()}</span><div style="font-size: 30px; font-weight: 800; line-height: 1.15">{big}</div><span style="font-size: 14px; opacity: 0.9">{sub}</span></div>'
    return f'<div style="display: grid; grid-template-columns: repeat({2 if mobile else 4}, minmax(0, 1fr)); gap: 12px">{"".join(t(l,b,s(),g) for l,b,s,g in items)}</div>'

def rivalries(mobile=False):
    out=""
    for a,b,n,wa,wb,note in RIVALS:
        out+=f'''<div style="padding: 16px 18px; border-radius: 12px; border: 1px solid {LINE}; display: flex; flex-direction: column; gap: 10px">
<span style="font-size: 13px; font-weight: 800; letter-spacing: 1px; color: {INK2}">{n} MEETINGS</span>
<div style="display: flex; justify-content: space-between; align-items: center"><span style="display: flex; flex-direction: column"><b style="color: {NAVY}; font-size: 18px">[Team {a}]</b><b style="font-size: 32px; line-height: 1; color: {NAVY if wa>=wb else WOLF}">{wa}</b></span><span style="font-weight: 800; color: {WOLF}">VS.</span><span style="display: flex; flex-direction: column; align-items: flex-end"><b style="color: {NAVY}; font-size: 18px">[Team {b}]</b><b style="font-size: 32px; line-height: 1; color: {NAVY if wb>=wa else WOLF}">{wb}</b></span></div>
<span style="display: flex; height: 10px; border-radius: 5px; overflow: hidden"><span style="flex: {wa} 1 0; background: {NAVY}"></span><span style="flex: {wb} 1 0; background: {WOLF}"></span></span>
<span style="font-size: 14px; color: {INK2}">{note}</span></div>'''
    return f'<div style="display: grid; grid-template-columns: repeat({1 if mobile else 3}, minmax(0, 1fr)); gap: 12px">{out}</div>'

def playoffs(mobile=False):
    rec=lambda t:(sum(1 for _,r in PO[t] if r in ("W","CW")),sum(1 for _,r in PO[t] if r in ("L","CL")))
    order=sorted(TEAMS,key=lambda t:(-rec(t)[0],rec(t)[1]))
    mx=max(len(PO[t]) for t in TEAMS) or 1
    def blk(y,r,first):
        bg={"W":"#1F8A4C","L":"#D9534E","CW":GOLD,"CL":SILVER}[r]; edge={"CW":GOLD_EDGE,"CL":SILVER_EDGE}.get(r,"none")
        return f'<span title="{y}" style="width: {100/mx:.2f}%; margin-left: {6 if first else 0}px; display: flex; align-items: center; justify-content: center; border-radius: 3px; background: {bg}; box-shadow: {edge}; color: {NAVY if r in ("CW","CL") else "#FFFFFF"}; font-size: 10px; font-weight: 800">&#39;{str(y)[2:]}</span>'
    def bar(t):
        out=""; prev=None
        for y,r in PO[t]:
            out+=blk(y,r,prev is not None and y!=prev); prev=y
        return out or f'<span style="font-size: 13px; color: {INK3}">Never made it</span>'
    rs="".join(f'''<div style="display: grid; grid-template-columns: 70px minmax(0, 1fr) 54px; gap: 10px; align-items: center; padding: 5px 0">
<b style="color: {NAVY}; font-size: 14px">Team {t}</b><span style="display: flex; height: 22px; gap: 2px">{bar(t)}</span>
<b style="text-align: right; font-size: 14px">{rec(t)[0]}-{rec(t)[1]}</b></div>''' for t in order)
    return card(rs+f'<div style="font-size: 13px; color: {INK2}; margin-top: 8px">Every playoff game, labeled by season. Green won, red lost; gold won the title, silver lost the final.</div>',mobile)

SILVER="linear-gradient(140deg, #FBFCFC 0%, #DDE2E5 30%, #B9C1C6 55%, #EEF1F3 78%, #A4ADB3 100%)"

SILVER_EDGE="inset 0 0 0 2px #8F989E, inset 0 1px 0 2px rgba(255,255,255,0.7), 0 2px 8px rgba(120,130,138,0.45)"

def superlatives(mobile=False, y=SEASONS[-1]):
    chips="".join(f'<button type="button" onclick="showYear(this, {yy})" style="font: inherit; cursor: pointer; padding: 7px 14px; border-radius: 18px; font-weight: 700; font-size: 14px; background: {NAVY if yy==y else "#FFFFFF"}; color: {"#FFFFFF" if yy==y else NAVY}; border: 1px solid {NAVY if yy==y else LINE}">{yy}</button>' for yy in SEASONS[::-1])
    good={"MVP","Steal of the draft","Best pickup","Best trade","Highest score","Best lineups"}
    tiles=""
    for lab,big,sub,team,stat in SUPER[y]:
        g=lab in good; mvp=lab=="MVP"
        bg=GOLD if mvp else NAVY; fg=NAVY if mvp else "#FFFFFF"
        person=sub!="" and not big[0].isdigit()
        tiles+=f'''<div style="display: flex; flex-direction: column; gap: 8px; padding: 16px 18px; border-radius: 12px; background: {bg}; color: {fg}{f"; box-shadow: {GOLD_EDGE}" if mvp else ""}">
<span style="font-size: 12px; font-weight: 800; letter-spacing: 1px; opacity: 0.8">{lab.upper()}</span>
<span style="display: flex; align-items: center; gap: 10px">{f'<span style="flex: none; border-radius: 50%; box-shadow: 0 0 0 2px #FFFFFF">{face(40, name=big)}</span>' if person else ""}<span style="display: flex; flex-direction: column; min-width: 0"><b style="font-size: {20 if person else 24}px; line-height: 1.1">{big}</b><span style="font-size: 13px; opacity: 0.8">{" · ".join(x for x in [sub, team if team not in big else ""] if x)}</span></span></span>
<span style="font-size: 15px; font-weight: 700">{stat}</span></div>'''
    return f'''<div style="display: flex; gap: 8px; flex-wrap: wrap">{chips}</div>
<div style="display: grid; grid-template-columns: repeat({2 if mobile else 4}, minmax(0, 1fr)); gap: 12px">{tiles}</div>'''

for y in SEASONS: HIST[y]["last"]=FINAL[y][-1]

AT=sorted([alltime(t) for t in TEAMS],key=lambda a:(-a["titles"],-a["W"]/(a["W"]+a["L"])))

def pf_ranks(t):
    out=[FINAL[y].index(t)+1 for y in SEASONS]
    out.append([r["t"] for r in rows].index(t)+1)
    return out

def medal(t,i):
    if i>=len(SEASONS): return None
    y=SEASONS[i]; return "G" if HIST[y]["champ"]==t else "S" if HIST[y]["runner"]==t else None

def vs_league(mobile=False):
    yrs=SEASONS+[2026]; cards=""
    avgf=lambda t: sum(pf_ranks(t)[:-1])/len(SEASONS)
    ordr=sorted(AT,key=lambda a:avgf(a["t"])); best_t=ordr[0]["t"]; worst_t=ordr[-1]["t"]
    W,Hh=200,90; px=lambda i: 14+i*(W-28)/(len(yrs)-1); py=lambda r: 10+(r-1)*(Hh-20)/11
    for a in ordr:
        t=a["t"]; rk=pf_ranks(t)
        pts=" ".join(f"{px(i):.1f},{py(r):.1f}" for i,r in enumerate(rk))
        dots="".join(f'<circle cx="{px(i):.1f}" cy="{py(r):.1f}" r="{7 if medal(t,i) else 5 if i==len(rk)-1 else 4}" fill="{"#FFC300" if medal(t,i)=="G" else "#B9C1C6" if medal(t,i)=="S" else SCALE[r-1]}" stroke="{NAVY if i==len(rk)-1 else INK3}" stroke-width="{2 if i==len(rk)-1 else 1}"/>' for i,r in enumerate(rk))
        labs="".join(f'<text x="{px(i):.1f}" y="{Hh+12}" font-size="10" text-anchor="middle" fill="{INK3}">&#39;{str(y)[2:]}</text>' for i,y in enumerate(yrs))
        avg=sum(rk[:-1])/len(rk[:-1])
        cards+=f'''<div style="padding: 12px 14px; border: 1px solid {LINE}; border-radius: 12px; display: flex; flex-direction: column; gap: 6px">
<span style="display: flex; justify-content: space-between; align-items: baseline"><b style="color: {NAVY}">[Team {t}]{" 🐐" if t==best_t else " 🗑️" if t==worst_t else ""}</b><span style="font-size: 13px; color: {INK2}">avg finish <b style="color: {NAVY}">{avg:.1f}</b></span></span>
<svg viewBox="0 0 {W} {Hh+16}" width="100%" aria-hidden="true"><line x1="8" x2="{W-8}" y1="{py(6.5):.1f}" y2="{py(6.5):.1f}" stroke="{LINE}" stroke-dasharray="3 3"/><polyline points="{pts}" fill="none" stroke="{NAVY}" stroke-width="2"/>{dots}{labs}</svg></div>'''
    return f'''<div style="display: grid; grid-template-columns: repeat({2 if mobile else 4}, minmax(0, 1fr)); gap: 12px">{cards}</div>
<div style="font-size: 13px; color: {INK2}">Final finish each year, decided by the final, the 3rd place game, and the consolation games, 1st at the top. Gold dots won the title, silver lost the final. Dashed line is the middle of the league; the outlined dot is 2026 so far. Average covers finished seasons.</div>'''

def hband(mobile=False):
    return f'''<section style="position: relative; overflow: hidden; background: {NAVY}; color: #FFFFFF">
<span aria-hidden="true" style="position: absolute; right: {-16 if mobile else -24}px; top: {-20 if mobile else -56}px; font-size: {150 if mobile else 300}px; font-weight: 800; line-height: 1; color: rgba(165,172,175,0.08)">WFR</span>
<div style="position: relative; {"padding: 18px 16px 10px" if mobile else "max-width: 1240px; margin: 0 auto; padding: 24px 24px 14px"}; display: flex; flex-direction: column; gap: 8px">
<span style="font-size: {14 if mobile else 15}px; font-weight: 700; color: {WOLF}; letter-spacing: 1px">Grudges, on the record</span>
<h1 style="margin: 0; font-size: {44 if mobile else 64}px; line-height: 0.95; font-weight: 800">History</h1>
<p style="margin: 0; font-size: {15 if mobile else 18}px; color: #D5D9DB">{LEAGUE_NAME} · {SEASONS[0]} to 2026</p></div>
{waves(50 if mobile else 80, NAVY)}</section>'''

def history_page(mobile=False):
    sec=lambda t,inner: f'<section style="display: flex; flex-direction: column; gap: 12px">{h2(t,"",mobile)}{inner}</section>'
    body=f'''<main style="{"padding: 4px 16px 32px" if mobile else "max-width: 1240px; margin: 0 auto; padding: 4px 24px 44px"}; display: flex; flex-direction: column; gap: {22 if mobile else 28}px">
{h_hero(mobile)}
{sec("Champions",champions(mobile))}
{sec("All-time standings",alltime_table(mobile))}
{sec("Head to head",h2h(mobile))}
{sec("Record book",records(mobile))}
{sec("Rivalries",rivalries(mobile))}
{sec("Every finish",vs_league(mobile))}
{sec("Playoff record",playoffs(mobile))}
{sec("Season superlatives",superlatives_all(mobile))}

</main>'''
    root='width: 390px; min-height: 6400px' if mobile else 'min-height: 100%'
    return f'''<div style="{root}; background: #FFFFFF; color: {INK}; font-family: {F}; font-size: 15px">
{navbar(active="History", league=True, mobile=mobile)}{hband(mobile)}{body}{footer(mobile)}</div>'''

def superlatives_all(mobile=False):
    # One block per season, newest showing; the year chips swap which block is visible
    return '<div data-years style="display: flex; flex-direction: column; gap: 12px">' + "".join(
        f'<div data-year="{yy}" style="display: flex; flex-direction: column; gap: 12px"{"" if yy == SEASONS[-1] else " hidden"}>{superlatives(mobile, yy)}</div>'
        for yy in SEASONS[::-1]) + "</div>"

write_page("history.html", "History", history_page(), history_page(True), active="History")
print("History written")

## Check

Lists every page written and flags any placeholder that didn't get filled in.

In [ ]:
left = {}
for f in WRITTEN:
    html = (SITE / f).read_text(encoding="utf-8")
    hits = re.findall(r"\[Team [^\]]+\]|\{[A-Z_]+\}|Sample data|\bnan\b", html)
    if hits:
        left[f] = sorted(set(hits))[:5]
print(len(WRITTEN), "pages in", SITE.resolve())
print("Unfilled placeholders:", left or "none")
if not (SITE / "assets" / "wfr-lockup.png").exists():
    print("Add the logo files to site/assets (wfr-lockup.png, wfr-icon-64.png, and wfr-icon-180.png).")